**top!**

In [ ]:
from pathlib import Path

# Resolve paths relative to the repository root.
ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
elif not (ROOT_DIR / "notebooks").exists() and (ROOT_DIR.parent / "notebooks").exists():
    ROOT_DIR = ROOT_DIR.parent

DATA_DIR = ROOT_DIR / "data" / "raw"
RESULTS_DIR = ROOT_DIR / "results"
MODELS_DIR = ROOT_DIR / "models"
PDF_DIR = RESULTS_DIR / "main_experiment"
SAVED_MODEL_DIR = MODELS_DIR / "main_experiment"
PARAM_CSV_PATH = SAVED_MODEL_DIR / "pinn_parameters.csv"

BASE_DIR = DATA_DIR


**Library**

In [ ]:
import os, glob, re
import numpy as np
import torch, torch.nn as nn
import soundfile as sf
import math
import random
import pandas as pd
import matplotlib.pyplot as plt
import sys
import csv
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from collections import Counter, defaultdict
from torch.quasirandom import SobolEngine
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import calinski_harabasz_score
from scipy.stats   import f_oneway    # ANOVA F
from matplotlib.ticker import MaxNLocator
from matplotlib import font_manager as fm
#from librosa import stft, amplitude_to_db
from scipy.signal import find_peaks
from scipy.signal import savgol_filter
from scipy.signal import welch
from scipy import signal
import warnings
from scipy.ndimage import gaussian_filter1d
from scipy.integrate import solve_ivp
warnings.filterwarnings("ignore", category=DeprecationWarning)

**Global Seed**

In [ ]:
SEED = 1111
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)

**Global win_sec, overlap**

In [ ]:
WIN_SEC = 0.1
MY_OVERLAP = 0.0

**PDF 저장 및 폰트 설정**

In [ ]:
# Use matplotlib default bundled fonts instead of local font files.
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["mathtext.fontset"] = "dejavusans"
plt.rcParams["text.usetex"] = False

# PDF 저장 시 폰트 포함 (논문용)
plt.rcParams["pdf.fonttype"] = 42

# PDF 저장 경로
pdf_dir = PDF_DIR
os.makedirs(pdf_dir, exist_ok=True)  # 폴더가 없으면 자동 생성

# PDF 저장 함수
def save_plot_as_pdf(pdf_filename):
    plt.savefig(pdf_filename, format="pdf", dpi=300, bbox_inches="tight")


🏊**wav file load**

In [ ]:
@dataclass
class Record:
    idx: int
    t: np.ndarray
    x: np.ndarray
    fs: float
    group: str
    filename: str

# 파일명 맨 앞 '숫자_' 형태에서 숫자(id)를 추출. 실패 시 None
def _parse_prefix_id(filepath: str):
    name = os.path.basename(filepath)
    m = re.match(r"^(\d+)_", name)
    return int(m.group(1)) if m else None

def concat_group(base_folder, group='health', use_ids=None):
    patterns = {
        'health'  : 'health_wav',
        'crackle' : 'crackle_wav',
        'wheeze'  : 'wheeze_wav'
    }
    if group not in patterns:
        raise ValueError(f"지원되지 않는 group: {group}")

    target_dir = os.path.join(base_folder, patterns[group])
    wavs = sorted(
        glob.glob(os.path.join(target_dir, '*.wav')),
        key=lambda x: _parse_prefix_id(x) if _parse_prefix_id(x) is not None else 999999
        )
    if not wavs:
        raise RuntimeError(f'No wav files found for {group} in {target_dir}')

    # prefix id 기준으로 필터링
    if use_ids is not None:
        use_ids_set = set(use_ids)

        # id별로 그룹핑
        id_to_files = {}
        for w in wavs:
            prefix_id = _parse_prefix_id(w)
            if prefix_id is None:
                continue
            if prefix_id in use_ids_set:
                id_to_files.setdefault(prefix_id, []).append(w)

        missing = [i for i in use_ids if i not in id_to_files]
        if missing:
            print(f"[WARN] {group}: 요청한 id 중 파일이 없는 값: {missing}")

        # 요청한 id 순서를 유지하면서, 각 id 내부는 파일명 정렬
        chosen = []
        for prefix_id in use_ids:
            if prefix_id in id_to_files:
                chosen.extend(sorted(id_to_files[prefix_id]))
    else:
        chosen = wavs

    # 로드
    result = []
    for idx, w in enumerate(chosen, start=1):
        x, fs = sf.read(w, dtype='float32')
        if x.ndim == 2:
            x = x.mean(axis=1)

        t = np.arange(len(x), dtype=np.float32) / fs
        filename = os.path.basename(w)
        prefix_id = _parse_prefix_id(w)

        print(f"[{group} #{idx}] id={prefix_id}  {filename:45s} → {len(x):,} samples ({len(x)/fs:.1f} s)")
        result.append(Record(
            idx=idx,
            t=t,
            x=x,
            fs=fs,
            group=group,
            filename=filename
        ))


    return result

# Call
BASE_SLICED_DIR = os.path.join(BASE_DIR, "sliced")

'''
H_IDS = [1, 4, 5, 6, 7, 10, 11, 15, 17, 18, 19, 20, 24, 26, 27]  # 15/28
W_IDS = [1, 2, 3, 4, 5, 6, 7]  # 7/8
C_IDS = [1, 2, 4, 5, 6, 7]  # 6/7
'''

H_IDS = [25]  # 1/28
W_IDS = [5]  # 1/8
C_IDS = [4]  # 3/7


file_H = concat_group(BASE_DIR, group='health',  use_ids=H_IDS)
file_W = concat_group(BASE_DIR, group='wheeze',  use_ids=W_IDS)
file_C = concat_group(BASE_DIR, group='crackle', use_ids=C_IDS)

**Plot: raw data smoothing**

In [ ]:
def smooth_raw_data(file_list, method='moving_avg', win_size=101, polyorder=3, pdf_dir=None, return_data=False):
    """
    file_list : concat_group 로드 결과 리스트 [(idx, t, x, fs, group), ...]
    method    : 'moving_avg' 또는 'savgol' 선택
    win_size  : smoothing window 크기
    polyorder : savgol 필터 차수 (method='savgol'일 때만 사용)
    """
    smoothed_list = []
    
    for record in file_list:
        idx = record.idx
        t = record.t
        x = record.x
        fs = record.fs
        group = record.group
        filename = record.filename
        if method == 'moving_avg':
            # 이동평균 스무딩
            win = np.ones(win_size) / win_size
            x_smooth = np.convolve(x, win, mode='same')
        elif method == 'savgol':
            # Savitzky–Golay 필터
            # win_size는 홀수여야 함
            if win_size % 2 == 0:
                win_size += 1
            x_smooth = savgol_filter(x, window_length=win_size, polyorder=polyorder)
        elif method == 'gaussian':
            # win_size를 sigma로 직접 쓰면 너무 세서, 조금 줄임
            sigma = win_size  
            x_smooth = gaussian_filter1d(x, sigma=sigma)
        else:
            raise ValueError("Unsupported method. Choose 'moving_avg' or 'savgol'.")

        # ---------- filename에서 원래 prefix 번호/seg 번호/구간 파싱 ----------
        # 예: 1_102_1b1_Ar_sc_Meditron_seg1_1.050-20.000.wav
        file_id = None
        seg_num = None
        seg_range = None

        m_id = re.match(r"^(\d+)_", filename)
        if m_id:
            file_id = m_id.group(1)  # '4' (문자열)

        m_seg = re.search(r"(seg\d+)_([0-9.]+-[0-9.]+)", filename)
        if m_seg:
            seg_num = m_seg.group(1)    # 'seg2'
            seg_range = m_seg.group(2)  # '7.100-20.000'

        # 파싱 실패 시 fallback
        if file_id is None:
            file_id = str(idx)  # 최후 fallback: 리스트 순번

        if seg_num is None or seg_range is None:
            file_tag = os.path.splitext(filename)[0]  # 확장자 제거 전체 이름
        else:
            # file4_seg2_7.100-20.000
            file_tag = f"file{file_id}_{seg_num}_{seg_range}"

        # Plot
        plt.figure(figsize=(16, 4))

        group_map = {'health': 'Healthy', 'wheeze': 'Wheeze', 'crackle': 'Crackle'}
        group_title = group_map.get(group.lower(), group)

        plt.plot(t, x, alpha=0.7, label='Raw')
        plt.plot(t, x_smooth, color='red', linewidth=1.0, label='Smoothed')
        '''
        if max_pts is not None and max_pts > 0:
            if len(t) > max_pts:
                idx_samp = np.linspace(0, len(t) - 1, num=max_pts, dtype=int)
                t_samp = t[idx_samp]
                x_samp = x_smooth[idx_samp]
                plt.scatter(t_samp, x_samp, color='black', s=10, alpha=0.6, label=f'Sampled ({max_pts})')'''

        duration_sec = len(x) / fs
        #plt.title(f"[{group_title} #{idx+1}]  fs={fs} Hz, {len(x)/fs:.1f} sec, win_size={win_size}")
        plt.title(f"Audio Waveform for {group_title} sound (Total Duration: {duration_sec:.0f} s)", fontsize=20)

        plt.xlabel("Time [s]", fontsize=24)
        plt.ylabel("Amplitude", fontsize=24)
        leg = plt.legend(fontsize=22, loc="upper right")
        leg.get_frame().set_alpha(0.4) 
        plt.xticks(fontsize=22)
        plt.yticks(fontsize=20)
        plt.ylim(-0.8, 0.8)
        plt.yticks(np.arange(-1.0, 1.1, 0.5))
        plt.gca().margins(x=0.025)
        plt.tight_layout()

        # save PDF
        '''
        if pdf_dir is None:
            pdf_dir = PDF_DIR  # 기본값
        os.makedirs(pdf_dir, exist_ok=True)
        pdf_filename = os.path.join(pdf_dir, f"{group}_{idx}_win{win_size}.pdf")
        save_plot_as_pdf(pdf_filename)
        '''
        plt.show()
        
        if return_data:
            smoothed_list.append(Record(
                idx=idx,
                t=t,
                x=x_smooth,
                fs=fs,
                group=group,
                filename=filename
            ))
            
    if return_data:
        return smoothed_list

# call
# Moving average smoothing

sm_H = smooth_raw_data(file_H, method='moving_avg', win_size=2500, return_data=True)
sm_W = smooth_raw_data(file_W, method='moving_avg', win_size=2500, return_data=True)
sm_C = smooth_raw_data(file_C, method='moving_avg', win_size=2500, return_data=True)

'''
sm_H = smooth_raw_data(file_H, method='gaussian', win_size=602, return_data=True, max_pts=441)
sm_W = smooth_raw_data(file_W, method='gaussian', win_size=602, return_data=True, max_pts=441)
sm_C = smooth_raw_data(file_C, method='gaussian', win_size=602, return_data=True, max_pts=441)'''

**Split window**

In [ ]:
def split_windows(t_cat, x_cat, fs_ref, win_sec, overlap=0.0,
                  target_pts=None, short_policy="drop"):
    """
    target_pts: 각 윈도우를 최종적으로 동일 길이로 만들고 싶을 때 (예: 441)
                - None이면 원본 샘플 개수를 그대로 사용
                - 값이 있으면 '균등 샘플링'으로 길이를 맞춤
    short_policy: win_sec보다 짧은 seg 처리
        - "drop": 버림 (빈 윈도우 반환)
        - "full": seg 전체를 1개 윈도우로 사용
        - "pad" : win_samples까지 0-padding 후 1개 윈도우
    """
    win_samples = int(round(win_sec * fs_ref))
    stride_samples = int(round(win_sec * (1 - overlap) * fs_ref))  # 다음 window 로 넘어갈 때 몇 샘플만큼 이동할지

    # --- 짧은 seg 처리 ---
    if len(x_cat) < win_samples:
        if short_policy == "drop":
            return np.empty((0, 0, 1), np.float32), np.empty((0, 0, 1), np.float32)
        elif short_policy == "full":
            t0 = (t_cat - t_cat[0]).astype(np.float32)
            x0 = x_cat.astype(np.float32)
            # 길이 맞추기(선택)
            if target_pts is not None:
                idx = np.linspace(0, len(t0) - 1, num=min(target_pts, len(t0)), dtype=int)
                t0, x0 = t0[idx], x0[idx]
            return t0[None, :, None], x0[None, :, None]
        elif short_policy == "pad":
            pad_len = win_samples - len(x_cat)
            x_pad = np.pad(x_cat, (0, pad_len), mode="constant")
            t_pad = np.arange(len(x_pad), dtype=np.float32) / fs_ref
            t0 = (t_pad - t_pad[0]).astype(np.float32)
            x0 = x_pad.astype(np.float32)
            if target_pts is not None and target_pts < len(t0):
                idx = np.linspace(0, len(t0) - 1, num=target_pts, dtype=int)
                t0, x0 = t0[idx], x0[idx]
            return t0[None, :, None], x0[None, :, None]
        else:
            raise ValueError("short_policy must be one of ['drop','full','pad']")

    # --- 일반 윈도우 분할 ---
    starts = range(0, len(x_cat) - win_samples + 1, stride_samples)  # 윈도우를 자를 시작점 index
    t_windows, x_windows = [], []

    for s in starts:
        t0 = (t_cat[s:s+win_samples] - t_cat[s]).astype(np.float32)
        x0 = x_cat[s:s+win_samples].astype(np.float32)

        if target_pts is not None and target_pts < len(t0):
            idx = np.linspace(0, len(t0) - 1, num=target_pts, dtype=int)
            t0, x0 = t0[idx], x0[idx]

        t_windows.append(t0[:, None])
        x_windows.append(x0[:, None])

    return np.stack(t_windows, axis=0), np.stack(x_windows, axis=0)


# 스무딩된 결과로 split windows
# max_pts: 441, 882, ..., None(4410) 중에서 적절한 값 무엇인지 확인해 보아야 함
window_H = [
    (*split_windows(rec.t, rec.x, rec.fs,
                    win_sec=WIN_SEC, overlap=MY_OVERLAP,
                    target_pts=441, short_policy="drop"),
     rec.fs, 'H', rec.filename)
    for rec in sm_H
]

window_W = [
    (*split_windows(rec.t, rec.x, rec.fs,
                    win_sec=WIN_SEC, overlap=MY_OVERLAP,
                    target_pts=441, short_policy="drop"),
     rec.fs, 'W', rec.filename)
    for rec in sm_W
]

window_C = [
    (*split_windows(rec.t, rec.x, rec.fs,
                    win_sec=WIN_SEC, overlap=MY_OVERLAP,
                    target_pts=441, short_policy="drop"),
     rec.fs, 'C', rec.filename)
    for rec in sm_C
]

print("\n[Healthy windows]")
for (t_win, x_win, fs, tag, filename) in window_H:
    print(f"{tag} | {filename:60s} | t_win={t_win.shape}  x_win={x_win.shape}")

print("\n[Wheeze windows]")
for (t_win, x_win, fs, tag, filename) in window_W:
    print(f"{tag} | {filename:60s} | t_win={t_win.shape}  x_win={x_win.shape}")

print("\n[Crackle windows]")
for (t_win, x_win, fs, tag, filename) in window_C:
    print(f"{tag} | {filename:60s} | t_win={t_win.shape}  x_win={x_win.shape}")


**Normalize & Standardization**

In [ ]:
# time 전역 min-max
def t_minmax(t_windows, eps=1e-9):
  t_min = t_windows.min()
  t_max = t_windows.max()
  minmax_norm = (t_windows - t_min) / (t_max - t_min + eps)
  return minmax_norm

# time 윈도우별 min-max -> 윈도우 시간을 0~0.03으로 local화 했기 때문에 전역과 윈도우별이 동일함

# amplitude 전역 z-score
'''def x_zscore(x_windows, eps=1e-9):
  mean = x_windows.mean()
  std  = x_windows.std() + eps
  zscore_norm = (x_windows - mean) / std
  return zscore_norm'''

# amplitude 윈도우별 z-score
def x_zscore_per_window(x_windows, eps=1e-9, return_stats=False):
  mean = x_windows.mean(axis=1, keepdims=True)
  std  = x_windows.std(axis=1, keepdims=True) + eps
  zscore_norm_per_win = (x_windows - mean) / std
  if return_stats:
    return zscore_norm_per_win, mean.squeeze(-1), std.squeeze(-1)   # 윈도우별 mean, std도 반환
  return zscore_norm_per_win

**Omega FFT + smoothing**

In [ ]:
def omega_fft(t, x, fmax=None):
    # 디트렌드
    x1 = x - x.mean()

    # 한닝 창(누설 완화)
    w  = np.hanning(len(x1))
    x2 = x1 * w

    # FFT
    dt    = np.mean(np.diff(t))
    Xf    = np.fft.rfft(x2)
    freqs = np.fft.rfftfreq(len(x2), dt)
    '''
    power = np.abs(Xf) ** 2
    f_centroid = np.sum(freqs * power) / np.sum(power)
    omega = 2.0 * np.pi * f_centroid

    '''
    amp   = np.abs(Xf)
    #print(len(amp))

    # --- Dominant frequency 추출 (DC 제외) ---
    if fmax is not None:
        mask = (freqs <= float(fmax))
        if np.any(mask):
            idx_domi = np.argmax(amp[mask])
            f_domi   = freqs[mask][idx_domi]
        else:
            # 마스크가 비면 전체 대역에서 탐색
            idx_domi = np.argmax(amp)
            f_domi   = freqs[idx_domi]
    else:
        idx_domi = np.argmax(amp)
        f_domi   = freqs[idx_domi]

    f_domi = freqs[idx_domi]

    # w(rad/s) 변환
    omega = 2.0 * np.pi * f_domi

    return omega

**B RMS Initialize**

In [ ]:
def B_rms(x):
     return float(np.sqrt(np.mean(x**2)))


**Build MLP - *Non-Linear Driven Damped Harmonic Oscillator***

In [ ]:
class Sine(nn.Module):
    """단순 sine 활성화"""
    def forward(self, x):
        return torch.sin(x)

class SirenLayer(nn.Module):
    """
    SIREN 논문 기반 레이어:
      - is_first=True 면 bound=1/n
      - else bound=√(6/n)/ω0
      - forward: sin(ω0 * (Wx + b))
    """
    def __init__(self, in_features, out_features, omega_0=1.0, is_first=False, bias=True):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self.init_weights()

    def init_weights(self):
        with torch.no_grad():
            n = self.linear.in_features
            if self.is_first:
                bound = 1.0 / n
            else:
                bound = math.sqrt(6.0 / n) / self.omega_0
            self.linear.weight.uniform_(-bound, bound)
            if self.linear.bias is not None:
                nn.init.zeros_(self.linear.bias)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))

# ── Fourier Encoder ─────────────────────────────────────────────
class FourierEncoder(nn.Module):
    def __init__(self, m: int = 16, sigma: float = 10.0):
        super().__init__()
        # (m, 1) 난수 주파수 행렬 - 고정 파라미터
        B = torch.randn(m, 1) * sigma
        self.register_buffer("B", B)         # 학습 X

    def forward(self, t: torch.Tensor) -> torch.Tensor:  # t: (N,1)
        proj = 2 * math.pi * t @ self.B.T                # (N,m)
        return torch.cat([proj.sin(), proj.cos()], dim=-1)  # (N,2m)

class MlpModel(nn.Module):
    def __init__(self,
                 in_dim,
                 n_hidden=3,
                 n_neurons=256,
                 activation='relu', # 여기에 relu, sine, siren, tanh 집어넣어!!!!!!!!!!!!!!!!!!!!!!
                 omega_0=1.0, # only used for siren
                 beta_init=None,
                 omega_init=None,  # FFT omega 기본값
                 B_init=None,  # RMS 기본값
                 fourier_m: int = 16, # 여기 수정하면 바봉 ㅎ
                 fourier_sigma: float = 10.0, # 여기 수정하면 바봉 ㅎ
                 use_fourier: bool = False,
                 scaling_factor = 1.0):
        super().__init__()

        self.scaling_factor = scaling_factor

        # ────────────────────  Fourier encoder ───────────────────
        self.use_fourier = use_fourier
        if use_fourier:
            self.enc = FourierEncoder(fourier_m, fourier_sigma)
            in_dim += 2 * fourier_m   # 입력 차원 확장
        else:
            self.enc = None
         # --------------------------------------------------------

        self.act = activation.lower()
        self.omega_0 = omega_0

        layers = []
        # 첫 번째 hidden 레이어
        if self.act == 'siren':
            layers.append(SirenLayer(in_dim, n_neurons, omega_0=self.omega_0, is_first=True))
        else:
            layers.append(nn.Linear(in_dim, n_neurons))
            layers.append(self._get_activation())

        # 중간 hidden 레이어들
        for _ in range(n_hidden - 1):
            if self.act == 'siren':
                layers.append(SirenLayer(n_neurons, n_neurons, omega_0=self.omega_0, is_first=False))
            else:
                layers.append(nn.Linear(n_neurons, n_neurons))
                layers.append(self._get_activation())

        # 출력 레이어 (항상 선형)
        layers.append(nn.Linear(n_neurons, 1))

        self.net = nn.Sequential(*layers)

        # PINN 파라미터

        # alpha 초기화
        alpha_init = 0.5
        self.log_alpha_hat = nn.Parameter(torch.log(torch.tensor([alpha_init/scaling_factor], dtype=torch.float32)))

        # beta 초기화
        self.log_beta_hat = nn.Parameter(torch.log(torch.tensor([beta_init/scaling_factor**2], dtype=torch.float32)))

        # omega 초기화
        self.log_omega_hat = nn.Parameter(torch.log(torch.tensor([omega_init/scaling_factor], dtype=torch.float32)))

        # B 초기화
        self.log_B_hat = nn.Parameter(torch.log(torch.tensor([B_init/scaling_factor**2], dtype=torch.float32)))

        # gamma 초기화
        r_init = 1.0
        self.r_hat = nn.Parameter(torch.tensor([r_init/scaling_factor**2], dtype=torch.float32))

        # phi 초기화
        self.phi  = nn.Parameter(torch.zeros(1))

        # 가중치 초기화 (non-siren)
        if self.act in ('relu', 'tanh', 'sine'):
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    if self.act == 'relu': # ReLU → He init
                        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                    else:
                        # tanh/sine 공통으로 Xavier 사용
                        gain = (1.0 if self.act == 'sine' # sine → gain=1.0
                                else nn.init.calculate_gain('tanh')) # tanh → gain=calculate_gain('tanh')
                        nn.init.xavier_uniform_(m.weight, gain=gain)
                    nn.init.zeros_(m.bias)

    def _get_activation(self):
        if self.act == 'relu':
            return nn.ReLU()
        elif self.act == 'tanh':
            return nn.Tanh()
        elif self.act == 'sine':
            return Sine()
        else:
            raise ValueError(f"Unsupported activation: {self.act}")

    #============== 양수 변환 ===============
    @property
    def alpha_hat(self):
        return torch.exp(self.log_alpha_hat)

    @property
    def beta_hat(self):
        return torch.exp(self.log_beta_hat)

    @property
    def omega_hat(self):
        return torch.exp(self.log_omega_hat)

    @property
    def B_hat(self):
        return torch.exp(self.log_B_hat)

    # ========== 물리 스케일로 복원 ==========
    @property
    def alpha(self):
        return self.alpha_hat * self.scaling_factor

    @property
    def beta(self):
        return self.beta_hat * (self.scaling_factor ** 2)

    @property
    def omega(self):
        return self.omega_hat * self.scaling_factor

    @property
    def r(self):
        return self.r_hat * (self.scaling_factor ** 2)

    @property
    def B(self):
        return self.B_hat * (self.scaling_factor ** 2)
    #============================================

    def forward(self, t):               # t shape (N, 1)
        if self.use_fourier:            # Fourier feature 사용 여부
            phi = self.enc(t)           # (N, 2m)
            x = torch.cat([t, phi], dim=-1)
        else:
            x = t
        return self.net(x)              # (N, 1)

**Loss Computation**

In [ ]:
#====================== Duffing Equation =================================
def loss_computation(tau, tau_colloc, model, u_obs, tau0, u0, u0_dot, tauT, uT, uT_dot,
                     lambda_data, lambda_pde, lambda_ic, lambda_bc):
  # Data loss
  u_pred_data = model(tau).reshape_as(u_obs)
  mse_data = nn.functional.mse_loss(u_pred_data, u_obs)

  # PDE loss
  tau_colloc = tau_colloc.clone().detach().requires_grad_(True)   # tau_colloc 자동미분 켜기
  u_pred_pde = model(tau_colloc)

  # 1차 derivative
  du_dtau = torch.autograd.grad(
      u_pred_pde, tau_colloc,
      grad_outputs=torch.ones_like(u_pred_pde),
      create_graph=True, retain_graph=True
      )[0]

  # 2차 derivative
  d2u_dtau2 = torch.autograd.grad(
      du_dtau, tau_colloc,
      grad_outputs=torch.ones_like(du_dtau),
      create_graph=True)[0]

  # driven eqn parameters
  alpha_hat = model.alpha_hat  # from @property
  beta_hat = model.beta_hat  # from @property
  omega_hat = model.omega_hat  # from @property
  phi = model.phi
  B_hat = model.B_hat
  r_hat = model.r_hat

  # Residual
  residual = d2u_dtau2 + alpha_hat * du_dtau + beta_hat * u_pred_pde + r_hat * u_pred_pde**3 - B_hat * torch.cos(omega_hat * tau_colloc + phi)

  # PDE loss (MSE_PDE)
  mse_pde = nn.functional.mse_loss(residual, torch.zeros_like(residual))

  # IC loss

  u0_pred = model(tau0).reshape_as(u0)
  du0_dtau = torch.autograd.grad(
      u0_pred, tau0,
      grad_outputs=torch.ones_like(u0_pred),
      create_graph=True)[0].reshape_as(u0_dot)

  mse_ic = nn.functional.mse_loss(u0_pred, u0) + nn.functional.mse_loss(du0_dtau, u0_dot)
  #mse_ic = torch.tensor(0.0, device=tau.device, dtype=tau.dtype)

  # BC loss
  uT_pred = model(tauT).reshape_as(uT)
  duT_dtau = torch.autograd.grad(
    uT_pred, tauT,
    grad_outputs=torch.ones_like(uT_pred),
    create_graph=True)[0].reshape_as(uT_dot)

  mse_bc = nn.functional.mse_loss(uT_pred, uT) + nn.functional.mse_loss(duT_dtau, uT_dot)

  # total loss
  total_loss = lambda_data * mse_data + lambda_pde * mse_pde + lambda_ic * mse_ic + lambda_bc * mse_bc

  scaling_factor = model.scaling_factor

  alpha = alpha_hat * scaling_factor
  beta = beta_hat * scaling_factor**2
  omega = omega_hat * scaling_factor
  r = r_hat * scaling_factor**2
  B = B_hat * scaling_factor**2

  return total_loss, mse_data, mse_pde, mse_ic, mse_bc, lambda_data, lambda_pde, lambda_ic, lambda_bc, u_pred_data, alpha, beta, omega, r, B

# lambda_ode weight curriculum
# ramp 방식
"""def get_lambda_pde_curriculum(ep, total_epochs, lambda_pde_max):
  progress = ep / max(total_epochs, 1)

  if progress <= 0.30:
    return 0.0
  elif progress <= 0.50:
    stage_progress = (progress - 0.30) / 0.20
    return lambda_pde_max * (0.25 * stage_progress)
  elif progress <= 0.70:
    stage_progress = (progress - 0.50) / 0.20
    return lambda_pde_max * (0.25 + 0.25 * stage_progress)
  elif progress <= 0.85:
    stage_progress = (progress - 0.70) / 0.15
    return lambda_pde_max * (0.50 + 0.25 * stage_progress)
  else:
    stage_progress = (progress - 0.85) / 0.15
    return lambda_pde_max * (0.75 + 0.25 * stage_progress)"""

# smoothstep 방식
def get_lambda_pde_curriculum(ep, total_epochs, lambda_pde_max):
  progress = min(max(ep / max(total_epochs, 1), 0.0), 1.0)

  floor_ratio = 0.02  # max 의 2% 로 시작
  plateau_start = 0.70  # 전체 학습 진행률 80% 시점부터 plateau 
  plateau_start = min(max(plateau_start, 1e-6), 1.0)

  if progress <= plateau_start:
    x = progress / plateau_start
    smooth = x * x * (3 - 2 * x)
    return lambda_pde_max * (floor_ratio + (1 - floor_ratio) * smooth)
  else:
    return lambda_pde_max

🐸**Configuration (cfg)**

In [ ]:
@dataclass
class TrainCfg:
  # MLP 구조
  n_hidden: int = 4
  n_neurons: int = 128

  # 학습 파라미터
  n_epochs: int = 5000
  lr: float = 1e-3
  n_colloc: int = 1000
  adaptive: bool = True
  warmup_ep: int = 500
  lambda_data: float = 1.0
  lambda_fixed: float = 0.05
  lambda_ic: float = 0.05
  lambda_bc: float = 0.05
  use_lambda_pde_curriculum: bool = False
  lambda_pde_max: float = 1.0
  lambda_min: float = 0.05
  #lambda_max: float = 1.0

  # optimizer
  optimizer: str = 'Adam'
  weight_decay: float = 1e-2   # AdamW (1e-4 ~ 1e-2)

  # L-BFGS
  use_lbfgs: bool = True  # True면 2-stage (AdamW + L-BFGS)
  lbfgs_epochs = 4  # 몇 번 호출할지 (8 / 40 / 120)
  lbfgs_max_iter = 25 # 한 step 안에서 line-search 몇 번 할지
  lbfgs_lr = 1.0
  lbfgs_history = 10
  # 총 LBFGS iter 계산 = epoch * max_iter

  # 기타
  print_every: int = 500
  use_best_state_adam: bool = True
  device: Optional[torch.device] = None
  #fmin: Optional[float] = None
  #fmax: Optional[float] = None

  # Early stop
  use_early_stop: bool = False
  es_patience: int = 100       # 20번 연속 개선 없으면 멈춤
  es_delta: float = 1e-4       # plateau로 간주할 최소 개선폭
  es_balance_tol: float = 1e-2 # | data_loss - λ·ode_loss | <= 1e-3
  es_eq_tol: float = 1e-12     # data_loss 와 λ·ode_loss 가 같다고 간주할 기준 (float 연산 특성 때문)
  es_min_ep: int = 300         # 최소 학습 epoch (이 값보다 작은 ep 에서는 early stop 무시)

# ----------!! 여기를 조정해 !!----------
cfg = TrainCfg(
    n_hidden=4,
    n_neurons=128,
    n_epochs=10000,
    lr=1e-3,
    n_colloc=1024,
    adaptive=False,
    warmup_ep=None,
    lambda_data = 1.0,
    lambda_fixed = 1.0,
    lambda_ic = 0.0,
    lambda_bc = 0.0,
    use_lambda_pde_curriculum = False,
    lambda_pde_max = 1.0,
    #lambda_max = 1.0,
    use_lbfgs=False,
    use_best_state_adam=True,
    print_every=100
  )

⚡ **Train Function Def - *Sobol colloc + Early Stop 추가***

In [ ]:
# 윈도우별 동일, ep마다는 다른 colloc point (랜덤이지만 간격 비슷하게)
def train_pinn(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    tau_train: torch.Tensor,
    u_train: torch.Tensor,
    *,
    cfg: TrainCfg,
    tau0,
    u0,
    u0_dot,
    tauT,
    uT,
    uT_dot):

    # 환경설정
    device = cfg.device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    tau_train = tau_train.to(device)
    u_train = u_train.to(device)
    tau0 = tau0.to(device)
    tauT = tauT.to(device)
    u0 = u0.to(device)
    uT = uT.to(device)
    u0_dot = u0_dot.to(device).unsqueeze(-1)
    uT_dot = uT_dot.to(device).unsqueeze(-1)

    # Early stop hyperparameter
    es_patience = cfg.es_patience        # plateau 허용 epoch
    es_delta = cfg.es_delta              # plateau로 간주할 최소 개선폭
    es_balance_tol = cfg.es_balance_tol  # |data - λ*ode| 허용 오차
    es_eq_tol = cfg.es_eq_tol            # data == λ*ode 판정 오차
    es_min_ep = cfg.es_min_ep            # 최소 학습 epoch
   
    history = []

    final_ep_u_pred = None

    # Plateau tracking
    best_total = float('inf')  # 관찰한 최소 total loss 저장
    wait = 0  # plateau 감지용 카운터 (개선 없을 때마다 +1)
    best_state_adam = None  # Adam/AdamW 단계에서 가장 성능 좋았던 시점 저장
    best_u_pred_np = None     # best_state_adam 일 때의 pred
    best_log = None
    stop_reason = None
    stopped = False

    # train loop (low-discrepancy colloc sampling)
    for ep in range(1, cfg.n_epochs + 1):
        sobol_ = SobolEngine(dimension=1, scramble=True, seed=SEED + ep)  # 매 에포크마다 SobolEngine을 새로 초기화해서 다른 시퀀스 사용
        sobol = sobol_.draw(cfg.n_colloc).to(device)   # [0,1] 구간에서 균일하게 퍼진 난수 벡터 뽑기 (cfg.n_colloc, 1)
        tau_colloc = tau0 + (tauT - tau0) * sobol

        '''
        # [DEBUG]
        if ep <= 1:
            print(f"[Epoch {ep}]", t_colloc[:1])
        '''
        if cfg.use_lambda_pde_curriculum:
            lambda_pde_current = get_lambda_pde_curriculum(ep, cfg.n_epochs, cfg.lambda_pde_max)
        else:
            lambda_pde_current = cfg.lambda_fixed

        # loss computation
        total_loss, mse_u, mse_pde, mse_ic, mse_bc, lambda_data, lambda_pde, lambda_ic, lambda_bc, u_pred_data, alpha, beta, omega, r, B = loss_computation(
            tau_train, tau_colloc, model, u_train, tau0, u0, u0_dot, tauT, uT, uT_dot, cfg.lambda_data, lambda_pde_current, cfg.lambda_ic, cfg.lambda_bc)

        # backpropagation, parameter update
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        # final epoch의 u_pred만 기록
        final_ep_u_pred = u_pred_data.detach().cpu().numpy().squeeze()
        total = total_loss.item() 

        # best snapshot tracking
        if total < best_total:
            best_total = total
            best_state_adam = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_u_pred_np = final_ep_u_pred

            """with torch.no_grad():
                lam_data_loss = lambda_data * mse_u.item()
                lam_pde_loss = lambda_pde * mse_pde.item()
                lam_ic_loss  = lambda_ic * mse_ic.item()
                lam_bc_loss  = lambda_bc * mse_bc.item()
                best_log = (
                    f"[{ep:5d}/{cfg.n_epochs}]\n"
                    f"------------------------------\n"
                    f" total_loss_adam(w)={total_loss.item():.2e} |\n"
                    f"------------------------------\n"
                    f" λ_data={lambda_data:.2e} |"
                    f" mse_u={mse_u.item():.2e} |"
                    f" λ*mse_u={lam_data_loss:.2e} |\n"
                    f"------------------------------------------------------------\n"
                    f" λ_pde={lambda_pde:.2e} |"
                    f" mse_pde={mse_pde.item():.2e} |"
                    f" λ*mse_pde={lam_pde_loss:.2e} |\n"
                    f"------------------------------------------------------------\n"
                    f" λ_IC={lambda_ic:.2e} |"
                    f" mse_IC={mse_ic.item():.2e} |"
                    f" λ_IC*mse_IC={lam_ic_loss:.2e} |\n"
                    f"------------------------------------------------------------\n"
                    f" λ_BC={lambda_bc:.2e} |"
                    f" mse_BC={mse_bc.item():.2e} |"
                     f" λ_BC*mse_BC={lam_bc_loss:.2e} |\n"
                    f"------------------------------------------------------------\n"
                    f" alpha={alpha.item():.2e} |"
                    f" beta={beta.item():.2e} |"
                    f" omega={omega.item():.2e} |"
                    f" γ={r.item():.2e} |"
                    f" B={B.item():.2e} |\n"
                    )"""

        history.append({
            'epoch': ep,
            'total_loss': total_loss.item(),
            'mse_u': mse_u.item(),
            'mse_pde': mse_pde.item(),
            'mse_IC': mse_ic.item(),
            'mse_BC': mse_bc.item(),
            'weighted_mse_u': lambda_data * mse_u.item(),
            'weighted_mse_pde': lambda_pde * mse_pde.item(),
            'weighted_mse_IC': lambda_ic * mse_ic.item(),
            'weighted_mse_BC': lambda_bc * mse_bc.item(),
            'lambda_data': lambda_data,
            'lambda_pde': lambda_pde,
            'lambda_IC': lambda_ic,
            'lambda_BC': lambda_bc,
            'alpha': alpha.item(),
            'beta': beta.item(),
            'omega': omega.item(),
            'r': r.item(),
            'B': B.item()
            #'u_pred_data': u_pred_data    # 마지막 epoch만 기록하도록 수정
        })

    """if best_log is not None:
        print(best_log, flush=True)"""

    if cfg.use_best_state_adam and best_state_adam is not None:
        model.load_state_dict(best_state_adam)
        if best_u_pred_np is not None:
            final_ep_u_pred = best_u_pred_np

    else:
        with torch.no_grad():
            lam_data_loss = lambda_data * mse_u.item()
            lam_pde_loss = lambda_pde * mse_pde.item()
            lam_ic_loss  = lambda_ic * mse_ic.item()
            lam_bc_loss  = lambda_bc * mse_bc.item()
            print(f"[{cfg.n_epochs:5d}/{cfg.n_epochs}] [Final Epoch]\n"
                f"------------------------------\n"
                f" total_loss_adam(w)={total_loss.item():.2e} |\n"
                f"------------------------------\n"
                f" λ_data={lambda_data:.2e} |"
                f" mse_u={mse_u.item():.2e} |"
                f" λ*mse_u={lam_data_loss:.2e} |\n"
                f"------------------------------------------------------------\n"
                f" λ_pde={lambda_pde:.2e} |"
                f" mse_pde={mse_pde.item():.2e} |"
                f" λ*mse_pde={lam_pde_loss:.2e} |\n"
                f"------------------------------------------------------------\n"
                f" λ_IC={lambda_ic:.2e} |"
                f" mse_IC={mse_ic.item():.2e} |"
                f" λ_IC*mse_IC={lam_ic_loss:.2e} |\n"
                f"------------------------------------------------------------\n"
                f" λ_BC={lambda_bc:.2e} |"
                f" mse_BC={mse_bc.item():.2e} |"
                f" λ_BC*mse_BC={lam_bc_loss:.2e} |\n"
                f"------------------------------------------------------------\n"
                f" alpha={alpha.item():.2e} |"
                f" beta={beta.item():.2e} |"
                f" omega={omega.item():.2e} |"
                f" γ={r.item():.2e} |"
                f" B={B.item():.2e} |\n",
                flush=True
                )

    print(f"[Completed] best_total={best_total:.3e}\n", flush=True)

    return history, final_ep_u_pred

⛹**Train One Window - *L-BFGS 추가***

In [ ]:
def train_one_window(t_input, u_input, cfg) -> Tuple[nn.Module, List[Dict[str, float]], np.ndarray]:
  t_raw = t_input.cpu().numpy().squeeze()
  x_raw = u_input.cpu().numpy().squeeze()

  #omega_init = omega_fft(t_raw, x_raw)  # omega fft + smoothing
  #omega_init = 2.0 * np.pi * 35.0
  omega_init = 2.0 * np.pi
  beta_init = omega_init**2
  #B_init = B_rms(x_raw)
  B_init = 0.5
  scaling_factor = 1 / WIN_SEC
  #scaling_factor = 1

  tau = scaling_factor * t_input # model 학습용
  tau_raw = tau.cpu().numpy().squeeze()

  h = float(np.diff(tau_raw).mean())  # 샘플 간격 [s]

  device = cfg.device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = MlpModel(
      in_dim = 1,
      n_hidden = cfg.n_hidden,
      n_neurons = cfg.n_neurons,
      beta_init = beta_init,
      omega_init = omega_init, # <-------------------- omega init 설정
      B_init = B_init,  # <-------------------- B init 설정
      fourier_m = 3,       # 피처 수  (8~32 권장)
      fourier_sigma = 1.0, # 주파수 스케일 (5~20 튜닝)
      use_fourier = False,
      scaling_factor = scaling_factor
      )

  model.to(device)
  '''
  # gamma를 physical 값 0.0으로 고정
  gamma_fixed = 0.0
  with torch.no_grad():
      model.r_hat.fill_(gamma_fixed / (scaling_factor ** 2))
  model.r_hat.requires_grad_(False)

  # B를 physical 값 0.5로 고정
  with torch.no_grad():
      model.log_B_hat.fill_(math.log(B_init / (scaling_factor ** 2)))
  model.log_B_hat.requires_grad_(False)
  '''
  alpha_init = model.alpha.item()
  r_init = model.r.item()
  phi_init = model.phi.item()
  
  tau_train = tau.to(device)
  u_train = u_input.to(device)

  print(f"omega_init: {model.omega.item():.4f}")
  print(f"beta_init: {model.beta.item():.4f}")
  print(f"B_init: {model.B.item():.4f}\n")

  # 1단계 optimizer
  if cfg.optimizer == "Adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
  elif cfg.optimizer == "AdamW":
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
  else:
    raise ValueError(f"Unsupported optimizer: {cfg.optimizer}")

  tau0 = tau_train[:1].clone().detach().to(device).requires_grad_(True)  # (1,1), τ=0 [soft IC]
  #tau0 = tau_train[:1].clone().detach().to(device) # [hard IC]
  tauT = tau_train[-1:].clone().detach().to(device).requires_grad_(True) # (1,1), τ=1
  u0 = u_train[:1].clone()    # (1,1)
  uT = u_train[-1:].clone()   # (1,1)

  # 도함수 타깃 (물리단위)
  if len(x_raw) >= 3:
      du0 = (-3*x_raw[0] + 4*x_raw[1] - x_raw[2]) / (2*h)
      duT = ( 3*x_raw[-1] - 4*x_raw[-2] + x_raw[-3]) / (2*h)
  else:
      du0 = (x_raw[1]  - x_raw[0])  / h
      duT = (x_raw[-1] - x_raw[-2]) / h
  u0_dot = torch.tensor([[du0]], device=device, dtype=u_train.dtype)
  uT_dot = torch.tensor([[duT]], device=device, dtype=u_train.dtype)

  #model.set_hard_ic(tau0, u0, u0_dot)

  print(
      "[IC/BC]:",
      f"u0={u0.detach().cpu().squeeze().item():.2e} |",
      f"u0'={u0_dot.detach().cpu().squeeze().item():.2e} |",
      f"uT={uT.detach().cpu().squeeze().item():.2e} |",
      f"uT'={uT_dot.detach().cpu().squeeze().item():.2e}\n",
      sep=" "
  )

  # 1단계 학습
  hist1, _ = train_pinn(
      model, optimizer,
      tau_train,
      u_train,
      cfg=cfg,
      tau0=tau0,
      u0=u0,
      u0_dot=u0_dot,
      tauT=tauT,
      uT=uT,
      uT_dot=uT_dot
  )

  adam_final_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
  lbfgs_failed = False
  lbfgs_fail_reason = ""

  # 2단계: L-BFGS
  hist2 = []

  if cfg.use_lbfgs:
    #device = cfg.device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    #model.to(device)
    #tau_train  = tau_train.to(device)
    #u_train  = u_train.to(device)

    lbfgs = torch.optim.LBFGS(model.parameters(),
                                  lr=cfg.lbfgs_lr,
                                  history_size=cfg.lbfgs_history,
                                  max_iter=cfg.lbfgs_max_iter,
                                  line_search_fn="strong_wolfe",
                                  tolerance_grad=1e-7,  # 현재 gradient의 norm이 이 값보다 작으면 기울기 0이라 판단 >> 거의 튜닝 X
                                  tolerance_change=1e-9,  # 최근 두 step의 파라미터(or loss) 변화가 이 값보다 작으면 더 이상 변하지 않는다 판단 >> 거의 튜닝 X
                                  )
    # line search: alpha(스텝 길이)를 얼마나 가야 손실이 줄어드는지 평가하며 찾는 절차
    # Strong Wolfe: 매 step 내부에서 여러 번 함수, gradient 평가하며 안정적인 alpha(스텝 길이) 자동 선택

    print("Starting L-BFGS ...")

    if cfg.use_lambda_pde_curriculum:
      lambda_pde_lbfgs = cfg.lambda_pde_max
    else:
      lambda_pde_lbfgs = cfg.lambda_fixed

    for step in range(1, cfg.lbfgs_epochs + 1):
      sobol_ = SobolEngine(dimension=1, scramble=True, seed=SEED + step)  # 매 에포크마다 SobolEngine을 새로 초기화해서 다른 시퀀스 사용
      sobol = sobol_.draw(cfg.n_colloc).to(device) # [0,1] 구간에서 균일하게 퍼진 난수 벡터 뽑기 (cfg.n_colloc, 1)
      tau_colloc = tau0 + (tauT - tau0) * sobol
      
      # [DEBUG]
      '''if step <= 1:
          print(f"[Epoch {step}]", tau_colloc[:1])'''

      def closure():
        lbfgs.zero_grad()
        total_loss, mse_u, mse_pde, mse_ic, mse_bc, lambda_data, lambda_pde, lambda_ic, lambda_bc, u_pred_data, alpha, beta, omega, r, B  = loss_computation(
            tau_train,
            tau_colloc,
            model,
            u_train,
            tau0,
            u0,
            u0_dot,
            tauT,
            uT,
            uT_dot,
            lambda_data=cfg.lambda_data,
            lambda_pde=lambda_pde_lbfgs,
            lambda_ic=cfg.lambda_ic,
            lambda_bc=cfg.lambda_bc,
        )
        total_loss.backward()

        # 외부에서 읽어올 수 있도록 값 보관
        closure.mse_u = mse_u.item()
        closure.mse_pde = mse_pde.item()
        closure.mse_ic = mse_ic.item()
        closure.mse_bc = mse_bc.item()
        closure.lambda_data = lambda_data
        closure.lambda_pde = lambda_pde
        closure.lambda_ic = lambda_ic
        closure.lambda_bc = lambda_bc
        closure.alpha = alpha.item()
        closure.beta = beta.item()
        closure.omega = omega.item()
        closure.r = r.item()
        closure.B = B.item()
        return total_loss

      loss_value = lbfgs.step(closure)
      loss_scalar = loss_value.item()
      lbfgs_is_finite = all([
          math.isfinite(loss_scalar),
          math.isfinite(closure.mse_u),
          math.isfinite(closure.mse_pde),
          math.isfinite(closure.mse_ic),
          math.isfinite(closure.mse_bc),
          math.isfinite(closure.alpha),
          math.isfinite(closure.beta),
          math.isfinite(closure.omega),
          math.isfinite(closure.r),
          math.isfinite(closure.B),
      ])

      if not lbfgs_is_finite:
        lbfgs_failed = True
        lbfgs_fail_reason = f"non-finite value at L-BFGS step {step}"
        model.load_state_dict(adam_final_state)
        hist2 = []
        print(f"[WARN] L-BFGS failed ({lbfgs_fail_reason}). Restored Adam final state.\n")
        break

      #if step % 10 == 0 or step ==1 or step == cfg.lbfgs_epochs:
      if step == cfg.lbfgs_epochs:
        lam_data_loss = closure.lambda_data * closure.mse_u
        lam_pde_loss = closure.lambda_pde * closure.mse_pde
        lam_ic_loss = closure.lambda_ic * closure.mse_ic
        lam_bc_loss = closure.lambda_bc * closure.mse_bc
        print(f"L-BFGS [{step}/{cfg.lbfgs_epochs}]\n"
              f"------------------------------\n"
              f" total_loss={loss_value.item():.2e} |\n"
              f"------------------------------------------------------------\n"
              f" λ_data={closure.lambda_data:.2e} |"
              f" mse_u={closure.mse_u:.2e} |"
              f" λ*mse_u={lam_data_loss:.2e} |\n"
              f"------------------------------------------------------------\n"
              f" λ_pde={closure.lambda_pde:.2e} |"
              f" mse_pde={closure.mse_pde:.2e} |"
              f" λ*mse_pde={lam_pde_loss:.2e} |\n"
              f"------------------------------------------------------------\n"
              f" λ_IC={closure.lambda_ic:.2e} |"
              f" mse_IC={closure.mse_ic:.2e} |"
              f" λ_IC*mse_IC={lam_ic_loss:.2e} |\n"
              f"------------------------------------------------------------\n"
              f" λ_BC={closure.lambda_bc:.2e} |"
              f" mse_BC={closure.mse_bc:.2e} |"
              f" λ_BC*mse_BC={lam_bc_loss:.2e} |\n"
              f"------------------------------------------------------------\n"
              f" alpha={closure.alpha:.2e} |"
              f" beta={closure.beta:.2e} |"
              f" omega={closure.omega:.2e} |"
              f" γ={closure.r:.2e} |"
              f" B={closure.B:.2e} |\n"
              )

      hist2.append({
          'epoch': len(hist1) + step,  # epoch 번호 이어쓰기
          'lambda_data': closure.lambda_data,
          'lambda_pde': closure.lambda_pde,
          'lambda_IC' : closure.lambda_ic,
          'lambda_BC' : closure.lambda_bc,
          'total_loss': loss_value.item(),
          'mse_u': closure.mse_u,
          'mse_pde': closure.mse_pde,
          'mse_IC' : closure.mse_ic,
          'mse_BC' : closure.mse_bc,
          'weighted_mse_u': closure.lambda_data * closure.mse_u,
          'weighted_mse_pde': closure.lambda_pde * closure.mse_pde,
          'weighted_mse_IC': closure.lambda_ic * closure.mse_ic,
          'weighted_mse_BC': closure.lambda_bc * closure.mse_bc,
          'alpha': closure.alpha,
          'beta': closure.beta,
          'omega': closure.omega,
          'r': closure.r,
          'B': closure.B
      })
    hist = hist1 if lbfgs_failed else (hist1 + hist2)
  else:
    hist = hist1

  with torch.no_grad():
    device_ = next(model.parameters()).device
    final_u_pred = model(tau.to(device_)).detach().cpu().numpy().squeeze()

  return model, hist, final_u_pred, omega_init, B_init, beta_init, alpha_init, r_init, phi_init

**Add ODE forward in prediction plot**

In [ ]:
def duffing_forward(t, u0, v0, alpha, beta, r, B, omega, phi=0.0, method="RK45", rtol=1e-7, atol=1e-9):
    """
    u'' + alpha*u' + beta*u + r*u^3 = B*cos(omega*tau + phi)
    """

    # RHS
    def rhs(ti, y):
        u, v = y  # unpacking
        du = v
        dv = (
            - alpha * v
            - beta  * u
            - r     * u**3
            + B * np.cos(omega * ti + phi)
        )
        return [du, dv]

    sol = solve_ivp(
        rhs,
        t_span=(float(t[0]), float(t[-1])),  # 적분 구간
        y0=[float(u0), float(v0)],  # 초기조건
        t_eval=t,  # 결과를 사용자가 원하는 시간 격자 t에서 반환하도록 지정
        method=method,  # Runge-Kutta 4(5)차 적응형 스텝 솔버 <- non-stiff에서 흔히 기본값
        rtol=rtol,  # 오차 허용치 (상대) -> 값이 작을수록 정확하지만 계산이 느려짐
        atol=atol,  # 오차 허용치(절대)
    )

    if not sol.success:
        raise RuntimeError(f"ODE solve failed: {sol.message}")

    return sol.y[0]   # u(t)

def fd_initial_conditions(t_raw, x_raw):
    dt = float(np.diff(t_raw).mean())   # t_raw는 [s]
    u0 = float(x_raw[0])

    if len(x_raw) >= 3:
        v0 = (-3*x_raw[0] + 4*x_raw[1] - x_raw[2]) / (2*dt)  # du/dt at t0
    else:
        v0 = (x_raw[1] - x_raw[0]) / dt

    return u0, float(v0)

def _to_numpy_1d(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy().squeeze()
    return np.asarray(x).squeeze()

def plot_ode_forward_v0_comparison(model, t_raw, x_raw, scaling_factor, window_idx=None, save_path=None, compare_pinn_autograd_v0=False):
    """
    Plot PINN prediction, ground truth, and ODE forward using GT u0 + GT finite-difference v0.
    Set compare_pinn_autograd_v0=True to additionally show the old mixed-IC comparison.
    """
    t_raw = _to_numpy_1d(t_raw).astype(np.float64).ravel()
    x_raw = _to_numpy_1d(x_raw).astype(np.float64).ravel()
    tau_raw = scaling_factor * t_raw

    was_training = model.training
    model.eval()

    try:
        device = next(model.parameters()).device
        dtype = next(model.parameters()).dtype

        # Latest hat-domain parameters from the current model.
        alpha = float(model.alpha_hat.item())
        beta  = float(model.beta_hat.item())
        omega = float(model.omega_hat.item())
        r     = float(model.r_hat.item())
        B     = float(model.B_hat.item())
        phi   = float(model.phi.item()) if hasattr(model, "phi") else 0.0

        u0, v0_gt_fd_dt = fd_initial_conditions(t_raw, x_raw)
        v0_gt_fd = float(v0_gt_fd_dt / scaling_factor)  # du/dtau

        tau_tensor = torch.as_tensor(tau_raw[:, None], device=device, dtype=dtype)
        with torch.no_grad():
            u_pinn = model(tau_tensor).detach().cpu().numpy().squeeze()

        v0_pinn_autograd = None
        if compare_pinn_autograd_v0:
            tau0 = torch.tensor([[float(tau_raw[0])]], device=device, dtype=dtype, requires_grad=True)
            u0_pred = model(tau0)
            v0_pinn_autograd = torch.autograd.grad(
                u0_pred,
                tau0,
                grad_outputs=torch.ones_like(u0_pred),
                create_graph=False,
                retain_graph=False,
            )[0].detach().cpu().item()

        print("[ODE forward IC check]")
        ic_msg = (
            f"window_idx={window_idx} | "
            f"u0={u0:.6e} | "
            f"v0_gt_fd={v0_gt_fd:.6e}"
        )
        if compare_pinn_autograd_v0:
            ic_msg += (
                f" | v0_pinn_autograd={v0_pinn_autograd:.6e} | "
                f"abs_diff={abs(v0_gt_fd - v0_pinn_autograd):.6e}"
            )
        print(ic_msg)

        ode_gt_fd = None
        ode_pinn_ad = None
        ode_gt_fd_msg = ""
        ode_pinn_ad_msg = ""

        try:
            ode_gt_fd = duffing_forward(
                t=tau_raw,
                u0=u0,
                v0=v0_gt_fd,
                alpha=alpha,
                beta=beta,
                r=r,
                B=B,
                omega=omega,
                phi=phi,
            )
        except Exception as e:
            ode_gt_fd_msg = str(e)
            print(f"[ODE FAIL] GT finite-diff v0 | window_idx={window_idx} | {e}")

        if compare_pinn_autograd_v0:
            try:
                ode_pinn_ad = duffing_forward(
                    t=tau_raw,
                    u0=u0,
                    v0=v0_pinn_autograd,
                    alpha=alpha,
                    beta=beta,
                    r=r,
                    B=B,
                    omega=omega,
                    phi=phi,
                )
            except Exception as e:
                ode_pinn_ad_msg = str(e)
                print(f"[ODE FAIL] PINN autograd v0 | window_idx={window_idx} | {e}")

        plt.figure(figsize=(8, 3))
        plt.plot(tau_raw, u_pinn, label="PINN pred", color="tab:blue")
        plt.plot(tau_raw, x_raw, "--", label="Ground Truth", color="tab:orange")

        if ode_gt_fd is not None:
            plt.plot(tau_raw, ode_gt_fd, color="tab:green", label="ODE forward (GT u0 + GT fd v0)")
        else:
            plt.text(0.5, 0.88, "ODE GT-fd v0 FAILED", transform=plt.gca().transAxes,
                     ha="center", color="tab:green", fontsize=10)

        if compare_pinn_autograd_v0:
            if ode_pinn_ad is not None:
                plt.plot(tau_raw, ode_pinn_ad, color="tab:red", label="ODE forward (GT u0 + PINN autograd v0)")
            else:
                plt.text(0.5, 0.78, "ODE PINN-autograd v0 FAILED", transform=plt.gca().transAxes,
                         ha="center", color="tab:red", fontsize=10)

        title_suffix = "v0 comparison" if compare_pinn_autograd_v0 else "ODE forward"
        title = f"window-{window_idx} prediction | {title_suffix}" if window_idx is not None else f"prediction | {title_suffix}"
        plt.xlabel("Tau", fontsize=14)
        plt.ylabel("Amplitude", fontsize=14)
        plt.title(title, fontsize=14)
        plt.legend(fontsize=10)
        plt.tick_params(axis="both", which="major", labelsize=14)
        ax = plt.gca()
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
        plt.tight_layout()

        if save_path is not None:
            save_plot_as_pdf(save_path)
        plt.show()
        plt.close()

        return {
            "tau_raw": tau_raw,
            "u_pinn": u_pinn,
            "ode_gt_fd": ode_gt_fd,
            "ode_pinn_autograd": ode_pinn_ad,
            "u0": u0,
            "v0_gt_fd": v0_gt_fd,
            "v0_pinn_autograd": v0_pinn_autograd,
            "alpha_hat": alpha,
            "beta_hat": beta,
            "omega_hat": omega,
            "r_hat": r,
            "B_hat": B,
            "phi": phi,
            "ode_gt_fd_msg": ode_gt_fd_msg,
            "ode_pinn_autograd_msg": ode_pinn_ad_msg,
        }
    finally:
        if was_training:
            model.train()


**Train All Windows**

In [ ]:
def train_all_windows(t_windows, x_windows, categories, cfg, num_train_windows=None, plot_n_windows=3, win_sec=WIN_SEC, file_idx=None, pdf_filename=None, models_dir=None):
  '''
  t_windows: List[torch.Tensor]
  x_windows: List[torch.Tensor]
  categories: List[str]
  plot_n_windows: 각 카테고리별로 plot할 window 개수
  '''
  # train all win 이 train all file 이 됨
  
  if num_train_windows is not None:  # num_train_win 필요없을 듯
      all_idx = list(range(len(t_windows)))
      random.shuffle(all_idx)
      train_idx = all_idx[:num_train_windows]

      t_windows = [t_windows[i] for i in train_idx]
      x_windows = [x_windows[i] for i in train_idx]
      categories = [categories[i] for i in train_idx]
  '''
  if num_train_windows is not None:  # num_train_win 필요없을 듯
    all_idx = list(range(len(t_windows)))
    #random.shuffle(all_idx)
    train_idx = all_idx[3:num_train_windows]

    t_windows = [t_windows[i] for i in train_idx]
    x_windows = [x_windows[i] for i in train_idx]
    categories = [categories[i] for i in train_idx]
    '''

    # win 자리에는 다 file 이 들어간다고 생각하기

  if models_dir is not None:
    os.makedirs(models_dir, exist_ok=True)

  histories = []

  alpha_list = []  # plot할 alpha
  beta_list = []   # plot
  omega_list = []  # plot
  r_list = []      # plot
  B_list = []      # plot
  phi_list = []

  alpha_hat_list = []  # plot할 alpha
  beta_hat_list = []   # plot
  omega_hat_list = []  # plot
  r_hat_list = []      # plot
  B_hat_list = []      # plot

  alpha_init_list = []
  beta_init_list = []
  omega_init_list = []
  r_init_list = []
  B_init_list = []
  phi_init_list = []

  cat_tags = []   # 카테고리 label
  win_ids = []   # 카테고리 내부 윈도우 번호 (1~666)
  final_u_preds = []   # 마지막 epoch u_pred만 기록
  t_list = []   # 각 window의 time 저장
  x_list = []   # 각 window의 ground truth 저장

  cat_counts = Counter(categories)   # 카테고리별 전체 개수
  #cat_summary = ", ".join(f"{cat}({cnt})" for cat, cnt in cat_counts.items())
  #print(f"\n====== {cat_summary} Training ======")

  cat_seen = defaultdict(int)   # 카테고리별 처리된 window 수
  cat_plotted = defaultdict(int)   # 카테고리별 플롯한 window 수

  def relu_no_utt_forward_train(t, u0, alpha, beta, r, B, omega, phi=0.0,
                                method="RK45", rtol=1e-7, atol=1e-9):
    """alpha*u_tau + beta*u + r*u^3 = B*cos(omega*tau + phi)."""
    t = np.asarray(t, dtype=np.float64).ravel()
    alpha = float(alpha)
    beta = float(beta)
    r = float(r)
    B = float(B)
    omega = float(omega)
    phi = float(phi)

    if np.isclose(alpha, 0.0):
      raise RuntimeError("alpha is too close to zero for ReLU no-u'' first-order forward")

    def rhs(ti, y):
      u = y[0]
      du = (B * np.cos(omega * ti + phi) - beta * u - r * u**3) / alpha
      return [du]

    sol = solve_ivp(
      rhs,
      t_span=(float(t[0]), float(t[-1])),
      y0=[float(u0)],
      t_eval=t,
      method=method,
      rtol=rtol,
      atol=atol,
    )

    if not sol.success:
      raise RuntimeError(f"ReLU no-u'' ODE solve failed: {sol.message}")

    return sol.y[0]

  for idx, (cat, t_input, x_input) in enumerate(zip(categories, t_windows, x_windows)):
    file_num = f"[File {file_idx}]"
    if cat_seen[cat] == 0:
      print(f"\n===== {file_num} {cat} training =====")

    cat_seen[cat] += 1
    print(f"{file_num}[{cat}] window {cat_seen[cat]}/{cat_counts[cat]}")

    model, hist, final_u_pred, omega_init, B_init, beta_init, alpha_init, r_init, phi_init = train_one_window(t_input, x_input, cfg)

    t_raw = t_input.cpu().numpy().squeeze()
    x_raw = x_input.cpu().numpy().squeeze()

    scaling_factor = 1.0 / win_sec
    tau_raw = scaling_factor * t_raw

    u0, v0 = fd_initial_conditions(t_raw, x_raw)
    v0 = v0 / scaling_factor   # du/dtau

    alpha = model.alpha_hat.item()
    beta  = model.beta_hat.item()
    omega = model.omega_hat.item()
    r     = model.r_hat.item()
    B     = model.B_hat.item()
    phi   = model.phi.item()

    try:
        u_sim_original = duffing_forward(
            t=tau_raw,
            u0=u0,
            v0=v0,
            alpha=alpha,
            beta=beta,
            r=r,
            B=B,
            omega=omega,
            phi=phi,
        )
        original_ode_success = True
    except Exception as e:
        print(f"[Original ODE FAIL] [{cat}] window {cat_seen[cat]} | {e}")
        u_sim_original = None
        original_ode_success = False

    try:
        u_sim_relu_no_utt = relu_no_utt_forward_train(
            t=tau_raw,
            u0=u0,
            alpha=alpha,
            beta=beta,
            r=r,
            B=B,
            omega=omega,
            phi=phi,
        )
        relu_no_utt_ode_success = True
    except Exception as e:
        print(f"[ReLU no-u'' ODE FAIL] [{cat}] window {cat_seen[cat]} | {e}")
        u_sim_relu_no_utt = None
        relu_no_utt_ode_success = False

    alpha_list.append(model.alpha.item())
    beta_list.append(model.beta.item())
    omega_list.append(model.omega.item())
    r_list.append(model.r.item())
    B_list.append(model.B.item())

    alpha_hat_list.append(model.alpha_hat.item())
    beta_hat_list.append(model.beta_hat.item())
    omega_hat_list.append(model.omega_hat.item())
    r_hat_list.append(model.r_hat.item())
    B_hat_list.append(model.B_hat.item())

    phi_list.append(model.phi.item())

    alpha_init_list.append(alpha_init)
    beta_init_list.append(beta_init)
    omega_init_list.append(omega_init)
    r_init_list.append(r_init)
    B_init_list.append(B_init)
    phi_init_list.append(phi_init)
    cat_tags.append(cat)
    win_ids.append(cat_seen[cat])   # 현재 카테고리의 local index

    # plot loss
    ep           = [h['epoch']           for h in hist]
    total_loss   = [h['total_loss']      for h in hist]
    data_w_loss  = [h['lambda_data'] * h['mse_u']   for h in hist]
    ode_w_loss   = [h['lambda_pde']  * h['mse_pde'] for h in hist]
    ic_loss      = [h['mse_IC']          for h in hist]

    # 모델 파라미터만 저장
    if models_dir is not None:
        model_path = os.path.join(models_dir, f"model_{file_idx}_{cat}_window_{cat_seen[cat]}.pt")
    else:
        model_path = f"model_{file_idx}_{cat}_window_{cat_seen[cat]}.pt"
    torch.save(model.state_dict(), model_path)
    print(f"[SAVE] {model_path}")

    del model
    torch.cuda.empty_cache()

    # history에 loss, mse_u, mse_pde만 저장
    histories.append({
        'loss': [h['total_loss'] for h in hist],
        'mse_u': [h['mse_u'] for h in hist],
        'mse_pde': [h['mse_pde'] for h in hist]
    })
    del hist   # 루프 내에서 사용이 끝난 변수 삭제
    torch.cuda.empty_cache()

    '''t_raw = t_input.cpu().numpy().squeeze()
    x_raw = x_input.cpu().numpy().squeeze()'''

    t_list.append(t_raw)
    x_list.append(x_raw)

    # 최종 예측값만 저장 (numpy 로 통일)
    if isinstance(final_u_pred, torch.Tensor):
      final_u_pred = final_u_pred.detach().cpu().numpy().squeeze()

    final_u_preds.append(final_u_pred)

    # 카테고리별로 plot_n_windows 수만큼 예측 곡선 plot
    # 샘플별 예측값이 맞는지 확인 (GT: t_raw, PINN: tau)
    if cat_plotted[cat] < plot_n_windows:
      # [DEBUG]
      #print(t_plot.shape, u_pred.shape)
      #print(u_pred[:10])  # 앞 10개 예측값 확인

      def _plot_prediction_with_ode(ode_values, ode_label, ode_success, ode_fail_text, file_tag, title_suffix):
        plt.figure(figsize=(8, 3))
        plt.plot(tau_raw, final_u_pred, label="PINN pred")
        plt.plot(tau_raw, x_raw, '--', label="Ground Truth")

        if ode_values is not None:
            plt.plot(tau_raw, ode_values, color="green", label=ode_label)
        else:
            plt.text(
                0.5, 0.9, ode_fail_text,
                transform=plt.gca().transAxes,
                ha="center", color="red", fontsize=12
            )

        title_msg = f"{cat} window-{cat_seen[cat]} prediction | {title_suffix}"
        if not ode_success:
            title_msg += " | ODE FAIL"

        plt.xlabel("Tau", fontsize=14)
        plt.ylabel("Amplitude", fontsize=14)
        plt.title(title_msg, fontsize=14)
        plt.legend(fontsize=12)
        plt.tick_params(axis='both', which='major', labelsize=14)
        ax = plt.gca()
        ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
        plt.tight_layout()

        if pdf_filename is not None:
                  fname = f"{file_tag}_file_{file_idx}_{cat}_window_{cat_seen[cat]}.pdf"
                  full_path = os.path.join(pdf_filename, fname)
                  save_plot_as_pdf(full_path)

        plt.show()
        plt.close()

      _plot_prediction_with_ode(
        ode_values=u_sim_original,
        ode_label="ODE forward (ReLU w/ u'')",
        ode_success=original_ode_success,
        ode_fail_text="ORIGINAL ODE FAILED",
        file_tag="original_ode",
        title_suffix="ODE (w/ u'')",
      )

      _plot_prediction_with_ode(
        ode_values=u_sim_relu_no_utt,
        ode_label="ODE forward (ReLU w/o u'')",
        ode_success=relu_no_utt_ode_success,
        ode_fail_text="ReLU no-u'' ODE FAILED",
        file_tag="relu_no_utt_ode",
        title_suffix="ODE (w/o u'')",
      )

      # plot loss
      '''
      plt.figure(figsize=(9,4.5))
      plt.semilogy(ep, raw_total_loss, '--', linewidth=2.0, alpha=0.9, label="total loss (raw)", zorder=3)
      plt.semilogy(ep, data_loss,  label="data loss", zorder=2)
      plt.semilogy(ep, ode_loss,   label="ODE loss", zorder=2)
      #plt.semilogy(ep, ic_loss,    label="IC loss")
      #plt.semilogy(ep, bc_loss,    label="BC loss")
      plt.xlabel("Epoch")
      plt.ylabel("Loss")
      plt.legend()
      plt.tight_layout()
      if pdf_filename is not None:
                fname = f"file_{file_idx}_{cat}_window_{cat_seen[cat]}.pdf"
                full_path = os.path.join(pdf_filename, f"loss_raw_{fname}")
                save_plot_as_pdf(full_path)
      plt.show()
      plt.close()
      '''
      
      # plot weighted loss
      plt.figure(figsize=(9,4.5))
      plt.semilogy(ep, total_loss,  '--', linewidth=2.0, alpha=0.9, label="total loss", zorder=3)
      plt.semilogy(ep, data_w_loss, label="λ·data loss", zorder=2)
      plt.semilogy(ep, ode_w_loss,  label="λ·ODE loss", zorder=2)
      #plt.semilogy(ep, ic_loss,     label="IC loss", zorder=2)
      plt.xlabel("Epoch")
      plt.ylabel("Loss")
      plt.legend()
      plt.tight_layout()
      if pdf_filename is not None:
              fname = f"file_{file_idx}_{cat}_window_{cat_seen[cat]}.pdf"
              full_path = os.path.join(pdf_filename, f"loss_lambda_{fname}")
              save_plot_as_pdf(full_path)
      plt.show()
      plt.close()

      cat_plotted[cat] += 1  # 해당 카테고리의 plot 개수 증가

      # Regression Metrics
      mse_val = mean_squared_error(x_raw, final_u_pred)
      mae_val = mean_absolute_error(x_raw, final_u_pred)
      rmse_val = np.sqrt(mse_val)
      r2_val  = r2_score(x_raw, final_u_pred)

      print(f"[Metrics – {cat} window {cat_seen[cat]}]")
      print(f"  MSE:  {mse_val:.4e}")
      print(f"  RMSE: {rmse_val:.4e}")
      print(f"  MAE:  {mae_val:.4e}")
      print(f"  R2:   {r2_val:.4f}\n")

  return histories, alpha_list, beta_list, omega_list, r_list, B_list, alpha_hat_list, beta_hat_list, omega_hat_list, r_hat_list, B_hat_list, phi_list, cat_tags, win_ids, final_u_preds, t_list, x_list, omega_init_list, B_init_list, beta_init_list, alpha_init_list, r_init_list, phi_init_list   #, models

**Helper Function**

In [ ]:
# NumPy → torch.Tensor 리스트 변환
def numpy_to_torch_list(t_arr, x_arr):
    t_list, x_list = [], []
    for t_win, x_win in zip(t_arr, x_arr):
        # (win_samples,1) 모양의 Tensor 로
        t_list.append(torch.from_numpy(t_win.astype('float32')).unsqueeze(-1))
        x_list.append(torch.from_numpy(x_win.astype('float32')).unsqueeze(-1))
    return t_list, x_list

**Train All Files**

In [ ]:
def train_all_files(file_windows_list, cfg, category_label, plot_n_windows=3):
    results = []
    for file_idx, file_data in enumerate(file_windows_list, start=1):

        t_windows, x_windows = file_data[:2]
        
        print(f"\n==== File {file_idx}/{len(file_windows_list)} ====")

        # train all file 필요 없어짐
        # results 에 저장 제대로 될지는 모르겠음

        # t min-max
        #t_windows_norm = t_minmax(t_windows)

        # x z-score
        #x_windows_norm, mu_list, std_list = x_zscore_per_window(x_windows, return_stats=True)

        # x |max|
        #x_windows_norm, max_vals = x_maxabs_per_window(x_windows)

        # NumPy → torch.Tensor 리스트 변환
        #t_win_norm_list, x_win_norm_list = numpy_to_torch_list(t_windows_norm, x_windows_norm)
        t_win_list, x_win_list = numpy_to_torch_list(t_windows, x_windows)
        #t_win_list, x_win_list = numpy_to_torch_list(t_windows, x_windows_norm)

        # 파일 식별용 catergory 태그
        #categories = [f"File{file_idx}"] * len(t_win_norm_list)
        categories = [category_label] * len(t_win_list)

        # 학습: train_all_windows 호출
        histories_, alpha_list_vals, beta_list_vals, omega_list_vals, r_list_vals, B_list_vals, alpha_hat_list_vals, beta_hat_list_vals, omega_hat_list_vals, r_hat_list_vals, B_hat_list_vals, phi_list_vals, cat_tags_, win_ids_, final_u_preds_, t_list, x_list, omega_init_vals, B_init_vals, beta_init_vals, alpha_init_vals, r_init_vals, phi_init_vals = train_all_windows(
            t_win_list,
            x_win_list,
            categories,
            cfg,
            num_train_windows=None,      # <------------------------------------------------- 학습할 윈도우 수 여기서 수정해 !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
            plot_n_windows=plot_n_windows,
            file_idx=file_idx,
            pdf_filename=pdf_dir,
            models_dir=SAVED_MODEL_DIR
        )

        # 결과 저장
        results.append({
            'file_idx': file_idx,
            'cat_tags': cat_tags_,
            'win_ids': win_ids_,
            'histories': histories_,

            # raw
            'alpha_list_vals': alpha_list_vals,
            'beta_list_vals': beta_list_vals,
            'omega_list_vals': omega_list_vals,
            'r_list_vals': r_list_vals,
            'B_list_vals': B_list_vals,

            # hat
            'alpha_hat_list_vals': alpha_hat_list_vals, 
            'beta_hat_list_vals': beta_hat_list_vals, 
            'omega_hat_list_vals': omega_hat_list_vals, 
            'r_hat_list_vals': r_hat_list_vals, 
            'B_hat_list_vals': B_hat_list_vals,

            'phi_list_vals': phi_list_vals,

            'omega_init_vals': omega_init_vals,
            'B_init_vals': B_init_vals,
            'beta_init_vals': beta_init_vals,
            'alpha_init_vals': alpha_init_vals,
            'r_init_vals': r_init_vals,
            'phi_init_vals': phi_init_vals,
            
            'final_u_preds': final_u_preds_,
            't_list': t_list,
            'x_list': x_list,
            #'max_vals': max_vals,
        })

    return results

**Run Train**

In [ ]:
file_windows_dict = {
    'H': window_H,
    'W': window_W,
    'C': window_C,
}

group_name_map = {
    'H': 'Healthy',
    'W': 'Wheeze',
    'C': 'Crackle',
}

# 결과 저장
results = {}  

for group_name, file_windows in file_windows_dict.items():  # file_windows 지워, file_windows_dict 도 무쓸모. group_name_map 이 클래스 구분해주는 루프
    label = group_name_map[group_name]
    results[group_name] = {}
    print(f"\n===== Training {group_name_map[group_name]} =====")

    train_results = train_all_files(file_windows, cfg, category_label=label, plot_n_windows=10) # <--- 여기서 plot_n_windows 이거도 파일별로 20s 짜리가 나와야 함. 수가 얼마 안되니까 인자 없애도 될듯
    # 여기 호출을 train all win 으로 해도 ㄱㅊ을듯
    # 그러면 위에 train all files 를 지우자

    results[group_name] = train_results

**2D Plot: Alpha, Beta**

In [ ]:
def plot_alpha_beta(alpha_list_vals, beta_list_vals,
                            cat_tags_, win_ids_, colors=None,
                            title="method 2 (Normalization)", pdf_filename=None):
    """
    α–β 산점도 (윈도우 번호 미표시)
    원본 plot_alpha_beta 와 동일한 인자 사용.
    """
    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for a, b, cat, _ in zip(alpha_list_vals, beta_list_vals, cat_tags_, win_ids_):
        ax.scatter(a, b, color=colors[cat], alpha=0.6)

    handles = [plt.Line2D([], [], marker='o', ls='', color=clr, label=lab)
               for lab, clr in colors.items()]

    leg = ax.legend(handles=handles, title="Class")
    leg.get_title().set_fontproperties(font_prop)

        # --- 축 라벨: 변수는 이탤릭, 나머지는 로만체 ---
    #ax.set_xlabel("\u03B1", fontproperties=font_prop_italic)
    #ax.set_ylabel("\u03B2", fontproperties=font_prop_italic)
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(r"$\beta$")

    ax.set_title(title, fontsize=18)

    ax.grid(True)
    plt.tight_layout()

    # PDF 저장 (파일명 전달받은 경우만 저장)
    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()


**2D Plot: Beta, B**

In [ ]:
def plot_beta_B(beta_list_vals, B_list_vals,
                            cat_tags_, win_ids_, colors=None,
                            title="method 2 (Normalization)", pdf_filename=None):
    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for a, b, cat, _ in zip(beta_list_vals, B_list_vals, cat_tags_, win_ids_):
        ax.scatter(a, b, color=colors[cat], alpha=0.6)

    handles = [plt.Line2D([], [], marker='o', ls='', color=clr, label=lab)
               for lab, clr in colors.items()]

    leg = ax.legend(handles=handles, title="Class")
    leg.get_title().set_fontproperties(font_prop)

    ax.set_xlabel(r"$\beta$")
    ax.set_ylabel(r"$B$")

    ax.set_title(title, fontsize=18)

    ax.grid(True)
    plt.tight_layout()

    # PDF 저장 (파일명 전달받은 경우만 저장)
    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()


**2D Plot: Alpha, Gamma**

In [ ]:
def plot_alpha_gamma(alpha_list_vals, r_list_vals,
                            cat_tags_, win_ids_, colors=None,
                            title="method 2 (Normalization)", pdf_filename=None):
    """
    α– Gamma 산점도 (윈도우 번호 미표시)
    원본 plot_alpha_beta 와 동일한 인자 사용.
    """
    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for a, b, cat, _ in zip(alpha_list_vals, r_list_vals, cat_tags_, win_ids_):
        ax.scatter(a, b, color=colors[cat], alpha=0.6)

    handles = [plt.Line2D([], [], marker='o', ls='', color=clr, label=lab)
               for lab, clr in colors.items()]

    leg = ax.legend(handles=handles, title="Class")
    leg.get_title().set_fontproperties(font_prop)

        # --- 축 라벨: 변수는 이탤릭, 나머지는 로만체 ---
    #ax.set_xlabel("\u03B1", fontproperties=font_prop_italic)
    #ax.set_ylabel("\u03B2", fontproperties=font_prop_italic)
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(r"$\gamma$")

    ax.set_title(title, fontsize=18)

    ax.grid(True)
    plt.tight_layout()

    # PDF 저장 (파일명 전달받은 경우만 저장)
    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()

**2D Plot: B, Omega**

In [ ]:
def plot_B_omega(B_list_vals, omega_list_vals,
                            cat_tags_, win_ids_, colors=None,
                            title="method 2 (Normalization)", pdf_filename=None):
    """
    B–Omega 산점도 (윈도우 번호 미표시)
    """
    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for a, b, cat, _ in zip(B_list_vals, omega_list_vals, cat_tags_, win_ids_):
        ax.scatter(a, b, color=colors[cat], alpha=0.6)

    handles = [plt.Line2D([], [], marker='o', ls='', color=clr, label=lab)
               for lab, clr in colors.items()]

    leg = ax.legend(handles=handles, title="Class")
    leg.get_title().set_fontproperties(font_prop)

    ax.set_xlabel(r"$B$")
    ax.set_ylabel(r"$\omega$")
    ax.set_title(title, fontsize=18)
    ax.grid(True)
    plt.tight_layout()

    # PDF 저장 (파일명 전달받은 경우만 저장)
    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()

**2D Plot: B, Gamma**

In [ ]:
def plot_B_gamma(B_list_vals, r_list_vals,
                            cat_tags_, win_ids_, colors=None,
                            title="method 2 (Normalization)", pdf_filename=None):
    """
    B–Gamma 산점도 (윈도우 번호 미표시)
    """
    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 6))
    ax = plt.gca()

    for a, b, cat, _ in zip(B_list_vals, r_list_vals, cat_tags_, win_ids_):
        ax.scatter(a, b, color=colors[cat], alpha=0.6)

    handles = [plt.Line2D([], [], marker='o', ls='', color=clr, label=lab)
               for lab, clr in colors.items()]

    leg = ax.legend(handles=handles, title="Class")
    leg.get_title().set_fontproperties(font_prop)

    ax.set_xlabel(r"$B$")
    ax.set_ylabel(r"$\gamma$")
    ax.set_title(title, fontsize=18)
    ax.grid(True)
    plt.tight_layout()

    # PDF 저장 (파일명 전달받은 경우만 저장)
    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()

**3D Plot: Alpha, Beta, Omega**

In [ ]:
def plot_alpha_beta_omega(alpha_list_vals, beta_list_vals, omega_list_vals,
                                  cat_tags_, win_ids_, colors=None,
                                  title="method 2 (Normalization)", pdf_filename=None):

  if colors is None:
    colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

  fig = plt.figure(figsize=(6, 6))
  ax  = fig.add_subplot(111, projection='3d')

  for a, b, o, cat, _ in zip(alpha_list_vals, beta_list_vals, omega_list_vals, cat_tags_, win_ids_):
    ax.scatter(a, b, o, color=colors[cat], alpha=0.6)

  # 범례용 더미 점
  for lab, clr in colors.items():
      ax.scatter([], [], [], color=clr, label=lab, alpha=0.6)

  ax.set_xlabel("\u03B1", fontproperties=font_prop_italic)
  ax.set_ylabel("\u03B2", fontproperties=font_prop_italic)
  ax.set_zlabel("\u03C9", fontproperties=font_prop_italic)

  #ax.set_xlabel(r"$\alpha$")
  #ax.set_ylabel(r"$\beta$")
  #ax.set_zlabel(r"$\omega$")

  ax.set_title(title, fontsize=18)

  plt.subplots_adjust(left=0.05, right=0.55)
  ax.legend(title="Class", loc='upper left', bbox_to_anchor=(0.9, 1.0))

  plt.tight_layout()

  # PDF 저장 (파일명 전달받은 경우만 저장)
  if pdf_filename:
      save_plot_as_pdf(pdf_filename)

  plt.show()

**3D Plot: Alpha, Beta, Gamma**

In [ ]:
def plot_alpha_beta_gamma(alpha_list_vals, beta_list_vals, r_list_vals,
                                  cat_tags_, win_ids_, colors=None,
                                  title="method 2 (Normalization)", pdf_filename=None):

  if colors is None:
    colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

  fig = plt.figure(figsize=(6, 6))
  ax  = fig.add_subplot(111, projection='3d')

  for a, b, o, cat, _ in zip(alpha_list_vals, beta_list_vals, r_list_vals, cat_tags_, win_ids_):
    ax.scatter(a, b, o, color=colors[cat], alpha=0.6)

  # 범례용 더미 점
  for lab, clr in colors.items():
      ax.scatter([], [], [], color=clr, label=lab, alpha=0.6)

  ax.set_xlabel("\u03B1", fontproperties=font_prop_italic)
  ax.set_ylabel("\u03B2", fontproperties=font_prop_italic)
  ax.set_zlabel("\u03B3", fontproperties=font_prop_italic)

  #ax.set_xlabel(r"$\alpha$")
  #ax.set_ylabel(r"$\beta$")
  #ax.set_zlabel(r"$\gamma$")

  ax.set_title(title, fontsize=18)

  plt.subplots_adjust(left=0.05, right=0.55)
  ax.legend(title="Class", loc='upper left', bbox_to_anchor=(0.9, 1.0))

  plt.tight_layout()

  # PDF 저장 (파일명 전달받은 경우만 저장)
  if pdf_filename:
      save_plot_as_pdf(pdf_filename)

  plt.show()

**3D Plot: Alpha, B, Gamma**

In [ ]:
def plot_alpha_B_gamma(alpha_list_vals, B_list_vals, r_list_vals,
                                  cat_tags_, win_ids_, colors=None,
                                  title="method 2 (Normalization)", pdf_filename=None):

  if colors is None:
    colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

  fig = plt.figure(figsize=(6, 6))
  ax  = fig.add_subplot(111, projection='3d')

  for a, b, o, cat, _ in zip(alpha_list_vals, B_list_vals, r_list_vals, cat_tags_, win_ids_):
    ax.scatter(a, b, o, color=colors[cat], alpha=0.6)

  # 범례용 더미 점
  for lab, clr in colors.items():
      ax.scatter([], [], [], color=clr, label=lab, alpha=0.6)

  ax.set_xlabel("\u03B1", fontproperties=font_prop_italic)
  ax.set_ylabel("B", fontproperties=font_prop_italic)
  ax.set_zlabel("\u03B3", fontproperties=font_prop_italic)

  #ax.set_xlabel(r"$\alpha$")
  #ax.set_ylabel(r"$B$")
  #ax.set_zlabel(r"$\gamma$")

  ax.set_title(title, fontsize=18)

  plt.subplots_adjust(left=0.05, right=0.55)
  ax.legend(title="Class", loc='upper left', bbox_to_anchor=(0.9, 1.0))

  plt.tight_layout()

  # PDF 저장 (파일명 전달받은 경우만 저장)
  if pdf_filename:
      save_plot_as_pdf(pdf_filename)

  plt.show()

**3D Plot: Beta, Omega, Gamma**

In [ ]:
def plot_beta_omega_gamma(beta_list_vals, omega_list_vals, r_list_vals,
                         cat_tags_, win_ids_, colors=None,
                         title="Beta-Omega-Gamma 3D Scatter", pdf_filename=None):

    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    fig = plt.figure(figsize=(6, 6))
    ax  = fig.add_subplot(111, projection='3d')

    for b, o, r_, cat, _ in zip(beta_list_vals, omega_list_vals, r_list_vals, cat_tags_, win_ids_):
        ax.scatter(b, o, r_, color=colors[cat], alpha=0.6)

    # 범례용 빈 점
    for lab, clr in colors.items():
        ax.scatter([], [], [], color=clr, label=lab, alpha=0.6)

    ax.set_xlabel("\u03B2", fontproperties=font_prop_italic)
    ax.set_ylabel("\u03C9", fontproperties=font_prop_italic)
    ax.set_zlabel("\u03B3", fontproperties=font_prop_italic)

    #ax.set_xlabel(r"$\beta$")
    #ax.set_ylabel(r"$\omega$")
    #ax.set_zlabel(r"$\gamma$")

    ax.set_title(title, fontsize=18)

    plt.subplots_adjust(left=0.05, right=0.55)
    ax.legend(title="Class", loc='upper left', bbox_to_anchor=(0.9, 1.0))

    plt.tight_layout()

    # PDF 저장 (파일명 전달받은 경우만 저장)
    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()

**Plot Call**

In [ ]:
# 전부다 플롯

alpha_all, beta_all, omega_all, r_all, B_all, cat_tags_all, win_ids_all = [], [], [], [], [], [], []
for grp_key, train_results in results.items():
    cat_label = group_name_map[grp_key]
    for file_res in train_results:
        alphas = file_res['alpha_list_vals']
        betas  = file_res['beta_list_vals']
        omegas = file_res['omega_list_vals']
        r_vals = file_res['r_list_vals']
        B_vals = file_res['B_list_vals']
        wins   = file_res['win_ids']
        alpha_all.extend(alphas)
        beta_all.extend(betas)
        omega_all.extend(omegas)
        r_all.extend(r_vals)
        B_all.extend(B_vals)
        # 각 윈도우에 대해 카테고리 이름으로 태그 설정
        cat_tags_all.extend([cat_label] * len(alphas))
        win_ids_all.extend(wins)

plot_alpha_beta(
    alpha_list_vals=alpha_all,
    beta_list_vals=beta_all,
    cat_tags_=cat_tags_all,
    win_ids_=win_ids_all,
    title=r"Duffing Equation $\alpha$–$\beta$ scatter",
    pdf_filename=os.path.join(pdf_dir, "Noff_alpha_beta_all.pdf")
)

plot_beta_B(
    beta_list_vals=beta_all,
    B_list_vals=B_all,
    cat_tags_=cat_tags_all,
    win_ids_=win_ids_all,
    title=r"Duffing Equation $\beta$–$B$ scatter",
    pdf_filename=os.path.join(pdf_dir, "Noff_beta_B_all.pdf")
)

plot_alpha_gamma(
    alpha_list_vals=alpha_all,
    r_list_vals=r_all,
    cat_tags_=cat_tags_all,
    win_ids_=win_ids_all,
    title=r"Duffing Equation $\alpha$–$\gamma$ scatter",
    pdf_filename=os.path.join(pdf_dir, "Noff_alpha_gamma_all.pdf")
)

plot_B_omega(
    B_list_vals=B_all,
    omega_list_vals=omega_all,
    cat_tags_=cat_tags_all,
    win_ids_=win_ids_all,
    title=r"Duffing Equation $B$–$\omega$ scatter",
    pdf_filename=os.path.join(pdf_dir, "Noff_B_omega_all.pdf")
)

plot_B_gamma(
    B_list_vals=B_all,
    r_list_vals=r_all,
    cat_tags_=cat_tags_all,
    win_ids_=win_ids_all,
    title=r"Duffing Equation $B$–$\gamma$ scatter",
    pdf_filename=os.path.join(pdf_dir, "Noff_B_gamma_all.pdf")
)

plot_alpha_beta_omega(
    alpha_list_vals=alpha_all,
    beta_list_vals=beta_all,
    omega_list_vals=omega_all,
    cat_tags_=cat_tags_all,
    win_ids_=win_ids_all,
    title=r"Duffing Equation $\alpha$–$\beta$–$\omega$ scatter",
    pdf_filename=os.path.join(pdf_dir, "Noff_alpha_beta_omega_all.pdf")
)

plot_alpha_beta_gamma(
    alpha_list_vals=alpha_all,
    beta_list_vals=beta_all,
    r_list_vals=r_all,
    cat_tags_=cat_tags_all,
    win_ids_=win_ids_all,
    title=r"Duffing Equation $\alpha$–$\beta$–$\gamma$ scatter",
    pdf_filename=os.path.join(pdf_dir, "Noff_alpha_beta_gamma_all.pdf")
)

plot_alpha_B_gamma(
    alpha_list_vals=alpha_all,
    B_list_vals=B_all,
    r_list_vals=r_all,
    cat_tags_=cat_tags_all,
    win_ids_=win_ids_all,
    title=r"Duffing Equation $\alpha$–$B$–$\gamma$ scatter",
    pdf_filename=os.path.join(pdf_dir, "Noff_alpha_B_gamma_all.pdf")
)

plot_beta_omega_gamma(
    beta_list_vals=beta_all,
    omega_list_vals=omega_all,
    r_list_vals=r_all,
    cat_tags_=cat_tags_all,
    win_ids_=win_ids_all,
    title=r"Duffing Equation $\beta$–$\omega$–$\gamma$ scatter",
    pdf_filename=os.path.join(pdf_dir, "Noff_beta_omega_gamma_all.pdf")
)

# ---------------------------------------------
# 파일당 플롯
# ---------------------------------------------
max_n_files = max(len(v) for v in results.values())  # 각 그룹-파일 개수의 최댓값

for file_idx in range(max_n_files):
    # 이 배치(batch)에 해당하는 α, β, ω, γ, B 태그, 윈도우 id 저장용
    alpha_batch, beta_batch, omega_batch, r_batch, B_batch = [], [], [], [], []
    cat_tags_batch, win_ids_batch = [], []

    for grp_key, train_results in results.items():
        if file_idx >= len(train_results):      # 어떤 그룹은 파일 수가 적을 수 있음
            continue

        file_res = train_results[file_idx]      # ← Health-1, Wheeze-1, Crackle-1 … 순서

        alpha_batch.extend(file_res['alpha_list_vals'])
        beta_batch.extend(file_res['beta_list_vals'])
        omega_batch.extend(file_res['omega_list_vals'])
        r_batch.extend(file_res['r_list_vals'])
        B_batch.extend(file_res['B_list_vals'])
        win_ids_batch.extend(file_res['win_ids'])

        cat_label = group_name_map[grp_key]     # 'Health' / 'Wheeze' / 'Crackle'
        cat_tags_batch.extend([cat_label] * len(file_res['alpha_list_vals']))

    title_suffix = f"(File {file_idx+1})"
    pdf_suffix = f"(File {file_idx+1})"

    plot_alpha_beta(
        alpha_list_vals=alpha_batch,
        beta_list_vals=beta_batch,
        cat_tags_=cat_tags_batch,
        win_ids_=win_ids_batch,
        title=fr"Duffing Equation $\alpha$–$\beta$ scatter {title_suffix}",
        pdf_filename=os.path.join(pdf_dir, f"Noff_scatter_alpha_beta_file_{pdf_suffix}.pdf")
    )

    plot_beta_B(
        beta_list_vals=beta_batch,
        B_list_vals=B_batch,
        cat_tags_=cat_tags_batch,
        win_ids_=win_ids_batch,
        title=fr"Duffing Equation $\beta$–$B$ scatter {title_suffix}",
        pdf_filename=os.path.join(pdf_dir, f"Noff_scatter_beta_B_file_{pdf_suffix}.pdf")
    )

    plot_alpha_gamma(
        alpha_list_vals=alpha_batch,
        r_list_vals=r_batch,
        cat_tags_=cat_tags_batch,
        win_ids_=win_ids_batch,
        title=fr"Duffing Equation $\alpha$–$\gamma$ scatter {title_suffix}",
        pdf_filename=os.path.join(pdf_dir, f"Noff_scatter_alpha_gamma_file_{pdf_suffix}.pdf")
    )

    plot_B_omega(
        B_list_vals=B_batch,
        omega_list_vals=omega_batch,
        cat_tags_=cat_tags_batch,
        win_ids_=win_ids_batch,
        title=fr"Duffing Equation $B$–$\omega$ scatter {title_suffix}",
        pdf_filename=os.path.join(pdf_dir, f"Noff_scatter_B_omega_file_{pdf_suffix}.pdf")
    )

    plot_B_gamma(
        B_list_vals=B_batch,
        r_list_vals=r_batch,
        cat_tags_=cat_tags_batch,
        win_ids_=win_ids_batch,
        title=fr"Duffing Equation $B$–$\gamma$ scatter {title_suffix}",
        pdf_filename=os.path.join(pdf_dir, f"Noff_scatter_B_gamma_file_{pdf_suffix}.pdf")
    )

    plot_alpha_beta_omega(
        alpha_list_vals=alpha_batch,
        beta_list_vals=beta_batch,
        omega_list_vals=omega_batch,
        cat_tags_=cat_tags_batch,
        win_ids_=win_ids_batch,
        title=fr"Duffing Equation $\alpha$–$\beta$–$\omega$ scatter {title_suffix}",
        pdf_filename=os.path.join(pdf_dir, f"Noff_scatter_alpha_beta_omega_file_{pdf_suffix}.pdf")
    )

    plot_alpha_beta_gamma(
        alpha_list_vals=alpha_batch,
        beta_list_vals=beta_batch,
        r_list_vals=r_batch,
        cat_tags_=cat_tags_batch,
        win_ids_=win_ids_batch,
        title=fr"Duffing Equation $\alpha$–$\beta$–$\gamma$ scatter {title_suffix}",
        pdf_filename=os.path.join(pdf_dir, f"Noff_scatter_alpha_beta_gamma_file_{pdf_suffix}.pdf")
    )

    plot_alpha_B_gamma(
        alpha_list_vals=alpha_batch,
        B_list_vals=B_batch,
        r_list_vals=r_batch,
        cat_tags_=cat_tags_batch,
        win_ids_=win_ids_batch,
        title=fr"Duffing Equation $\alpha$–$B$–$\gamma$ scatter {title_suffix}",
        pdf_filename=os.path.join(pdf_dir, f"Noff_scatter_alpha_B_gamma_file_{pdf_suffix}.pdf")
    )

    plot_beta_omega_gamma(
        beta_list_vals=beta_batch,
        omega_list_vals=omega_batch,
        r_list_vals=r_batch,
        cat_tags_=cat_tags_batch,
        win_ids_=win_ids_batch,
        title=fr"Duffing Equation $\beta$–$\omega$–$\gamma$ scatter {title_suffix}",
        pdf_filename=os.path.join(pdf_dir, f"Noff_scatter_beta_omega_gamma_file_{pdf_suffix}.pdf")
    )

**Plot Histogram**

In [ ]:
def plot_coefficient_histogram(values, cat_tags, coeff_name,
                               colors=None, bins=30, init_values=None, pdf_filename=None
                               ):
    """
    단일 계수에 대한 클래스별 히스토그램
    - values   : 리스트 또는 1D array, 계수 값
    - cat_tags : 리스트, 각 값에 대응하는 클래스 레이블
    - coeff_name: str, 예: "Alpha", "Beta" 등 (xlabel, title에 사용)
    """
    if colors is None:
        colors = {'Healthy': 'blue', 'Wheeze': 'red', 'Crackle': 'green'}

    plt.figure(figsize=(6, 4))

    values_np = np.array(values)
    bin_edges = np.histogram_bin_edges(values_np, bins=bins)

    if coeff_name.startswith(r"$\beta$"):
        # Beta: 0~100000 사이를 bins 개수로 나눔
        x_min = values_np.min()
        x_max = 40
        bin_edges = np.linspace(x_min, x_max, bins + 1)  # 사용 안하는 중

    elif coeff_name.startswith(r"$\gamma$"):
        # Gamma: -40~0 사이를 bins 개수로 나눔
        x_min, x_max = -70, 5
        bin_edges = np.linspace(x_min, x_max, bins + 1)
    else:
        # 나머지 계수는 데이터 기반 자동 binning
        bin_edges = np.histogram_bin_edges(values_np, bins=bins)

    # 클래스별로 histogram
    for cls, clr in colors.items():
        # 해당 클래스에 속하는 값만 뽑아서 그리기
        vals = [v for v, tag in zip(values, cat_tags) if tag == cls]

        # 개수 → 확률로 바꾸기: 클래스당 총 개수 = vals 길이
        weights = np.ones_like(vals) / len(vals)
        plt.hist(vals, bins=bin_edges,
                    weights=weights,
                    alpha=0.6, label=cls, color=clr)
        
    # Initial value vertical dashed line
    if init_values is not None and len(init_values) > 0:
        init_val = init_values[0]  # 모든 window에서 동일한 초기값
        plt.axvline(init_val, color='gray', linestyle='--',
                    linewidth=2.0, alpha=0.8, label="Init value")

    plt.xlabel(coeff_name + " Values", fontsize=24)
    plt.ylabel("Probability", fontsize=24)
    #plt.title(f"{coeff_name}")
    plt.legend(fontsize=16, loc="upper center")
    plt.xticks(fontsize=22)
    plt.yticks(np.arange(0, 1.01, 0.1), fontsize=22)
    plt.grid(True, linestyle='--', alpha=0.3)
    #plt.ylim(0, 1)
    if coeff_name.startswith(r"$\gamma$"):
        plt.ylim(0, 0.6)
    else:
        plt.ylim(0, 0.3)
    #plt.gca().margins(x=0.02)

    '''if coeff_name.startswith(r"$\gamma$"):
        plt.xlim(-40, 0)'''
    '''
    if coeff_name.startswith("Beta"):
        plt.xlim(0, 100000)'''

    plt.tight_layout()

    # PDF 저장 (파일명 전달받은 경우만 저장)
    if pdf_filename:
        save_plot_as_pdf(pdf_filename)

    plt.show()
#==========================================================================================================================

# 전부다 플롯
alpha_all, beta_all, omega_all, r_all, B_all, cat_tags_all, alpha_init_all, beta_init_all, omega_init_all, r_init_all, B_init_all, file_ids_all = [], [], [], [], [], [], [], [], [], [], [], []

for grp_key, train_results in results.items():
    cat_label = group_name_map[grp_key]
    for file_res in train_results:
        file_idx = file_res['file_idx']
        n = len(file_res['alpha_list_vals'])
        alpha_all.extend(file_res['alpha_list_vals'])
        beta_all.extend(file_res['beta_list_vals'])
        omega_all.extend(file_res['omega_list_vals'])
        r_all.extend(file_res['r_list_vals'])
        B_all.extend(file_res['B_list_vals'])
        # 각 윈도우마다 동일한 클래스 라벨을 태깅
        cat_tags_all.extend([cat_label] * n)
        file_ids_all.extend([file_idx] * n)
        alpha_init_all.extend(file_res['alpha_init_vals'])
        beta_init_all.extend(file_res['beta_init_vals'])
        omega_init_all.extend(file_res['omega_init_vals'])
        r_init_all.extend(file_res['r_init_vals'])
        B_init_all.extend(file_res['B_init_vals'])

# ——————————————
# 각 계수에 대해 히스토그램 호출
plot_coefficient_histogram(alpha_all, cat_tags_all, r"$\alpha$", init_values=alpha_init_all, pdf_filename=os.path.join(pdf_dir, "Noff_histogram_alpha_all.pdf"))
plot_coefficient_histogram(alpha_init_all, cat_tags_all, r"$\alpha_{\mathrm{init}}$", pdf_filename=os.path.join(pdf_dir, "Noff_histogram_alpha_init_all.pdf"))
plot_coefficient_histogram(beta_all,  cat_tags_all, r"$\beta$", init_values=beta_init_all, pdf_filename=os.path.join(pdf_dir, "Noff_histogram_beta_all.pdf"))
plot_coefficient_histogram(beta_init_all, cat_tags_all, r"$\beta_{\mathrm{init}}$", pdf_filename=os.path.join(pdf_dir, "Noff_histogram_beta_init_all.pdf"))
plot_coefficient_histogram(omega_all, cat_tags_all, r"$\omega$", init_values=omega_init_all, pdf_filename=os.path.join(pdf_dir, "Noff_histogram_omega_all.pdf"))
plot_coefficient_histogram(omega_init_all, cat_tags_all, r"$\omega_{\mathrm{init}}$", pdf_filename=os.path.join(pdf_dir, "Noff_histogram_omega_init_all.pdf"))
plot_coefficient_histogram(r_all,     cat_tags_all, r"$\gamma$", bins=30, init_values=r_init_all,  pdf_filename=os.path.join(pdf_dir, "Noff_histogram_r_all.pdf"))
plot_coefficient_histogram(r_init_all, cat_tags_all, r"$\gamma_{\mathrm{init}}$", pdf_filename=os.path.join(pdf_dir, "Noff_histogram_r_init_all.pdf"))
plot_coefficient_histogram(B_all,     cat_tags_all, r"$B$", init_values=B_init_all, pdf_filename=os.path.join(pdf_dir, "Noff_histogram_B_all.pdf"))
plot_coefficient_histogram(B_init_all, cat_tags_all, r"$B_{\mathrm{init}}$", pdf_filename=os.path.join(pdf_dir, "Noff_histogram_B_init_all.pdf"))

# 파일당 플롯
max_n_files = max(len(v) for v in results.values())   # 각 그룹-파일 개수의 최댓값

for file_idx in range(max_n_files):
    # ── 1) 한 파일의 계수·태그 모으기 ─────────────────────────
    alpha_batch, beta_batch, omega_batch = [], [], []
    r_batch,    B_batch                = [], []
    cat_tags_batch, alpha_init_batch, beta_init_batch, omega_init_batch, r_init_batch, B_init_batch = [], [], [], [], [], []

    for grp_key, train_results in results.items():
        if file_idx >= len(train_results):          # 그룹마다 파일 수가 달라도 안전
            continue

        file_res  = train_results[file_idx]         # Health-1, Wheeze-1, Crackle-1 …

        alpha_batch.extend(file_res['alpha_list_vals'])
        beta_batch.extend(file_res['beta_list_vals'])
        omega_batch.extend(file_res['omega_list_vals'])
        r_batch.extend(file_res['r_list_vals'])
        B_batch.extend(file_res['B_list_vals'])
        alpha_init_batch.extend(file_res['alpha_init_vals'])
        beta_init_batch.extend(file_res['beta_init_vals'])
        omega_init_batch.extend(file_res['omega_init_vals'])
        r_init_batch.extend(file_res['r_init_vals'])
        B_init_batch.extend(file_res['B_init_vals'])

        cat_label = group_name_map[grp_key]         # 'Health' / 'Wheeze' / 'Crackle'
        cat_tags_batch.extend([cat_label] * len(file_res['alpha_list_vals']))

    # ── 2) 파일별 히스토그램 그리기 ────────────────────────────────────
    title_suffix = f"(File {file_idx + 1})"
    pdf_suffix = f"(File {file_idx+1})"
    title_alpha_init = r"$\alpha_{\mathrm{init}}$"
    title_beta_init = r"$\beta_{\mathrm{init}}$"
    title_omega_init = r"$\omega_{\mathrm{init}}$"
    title_r_init = r"$\gamma_{\mathrm{init}}$"
    title_B_init = r"$B_{\mathrm{init}}$"

    plot_coefficient_histogram(alpha_batch, cat_tags_batch, fr"$\alpha$ {title_suffix}", pdf_filename=os.path.join(pdf_dir, f"Noff_histogram_alpha_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(alpha_init_batch, cat_tags_batch, f"{title_alpha_init} {title_suffix}", pdf_filename=os.path.join(pdf_dir, f"Noff_histogram_alpha_init_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(beta_batch,  cat_tags_batch, fr"$\beta$ {title_suffix}", pdf_filename=os.path.join(pdf_dir, f"Noff_histogram_beta_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(beta_init_batch, cat_tags_batch, f"{title_beta_init} {title_suffix}", pdf_filename=os.path.join(pdf_dir, f"Noff_histogram_beta_init_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(omega_batch, cat_tags_batch, fr"$\omega$ {title_suffix}", pdf_filename=os.path.join(pdf_dir, f"Noff_histogram_omega_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(omega_init_batch, cat_tags_batch, f"{title_omega_init} {title_suffix}", pdf_filename=os.path.join(pdf_dir, f"Noff_histogram_omega_init_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(r_batch,     cat_tags_batch, fr"$\gamma$ {title_suffix}", bins=30, pdf_filename=os.path.join(pdf_dir, f"Noff_histogram_r_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(r_init_batch, cat_tags_batch, f"{title_r_init} {title_suffix}", pdf_filename=os.path.join(pdf_dir, f"Noff_histogram_r_init_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(B_batch,     cat_tags_batch, fr"$B$ {title_suffix}", pdf_filename=os.path.join(pdf_dir, f"Noff_histogram_B_file_{pdf_suffix}.pdf"))
    plot_coefficient_histogram(B_init_batch, cat_tags_batch, f"{title_B_init} {title_suffix}", pdf_filename=os.path.join(pdf_dir, f"Noff_histogram_B_init_file_{pdf_suffix}.pdf"))


**Evaluation Metric**

In [ ]:
n = len(alpha_all)  # 또는 len(r_all), 윈도우 개수
alpha0 = 0.5
r0 = 1.0
alpha_init_all = [alpha0] * n
r_init_all     = [r0] * n

def fisher_discriminant_ratio(vals, labels):
    """FDR: (클래스 평균 분산) / (클래스 내부 분산)"""
    classes = np.unique(labels)
    means = [vals[labels == c].mean() for c in classes]
    variances = [vals[labels == c].var(ddof=1) for c in classes]
    between = np.var(means, ddof=1)
    within  = np.mean(variances)
    return between / within if within > 0 else np.nan

def anova_F(vals, labels):
    """ANOVA F-statistic"""
    groups = [vals[labels == c] for c in np.unique(labels)]
    # 그룹 중 하나라도 샘플 수가 1개 이하면 F 계산 불가 → nan
    if any(len(g) < 2 for g in groups):
        return np.nan
    # 각 그룹이 전부 같은 값(상수)이라면 F 계산 불가 → nan
    if all(np.all(g == g[0]) for g in groups):
        return np.nan
    return f_oneway(*groups).statistic

def anova_p_value(vals, labels):
    """ANOVA p-value"""
    groups = [vals[labels == c] for c in np.unique(labels)]
    # 그룹 중 하나라도 샘플 수가 1개 이하면 p-value 계산 불가 → nan
    if any(len(g) < 2 for g in groups):
        return np.nan
    # 각 그룹이 전부 같은 값(상수)이라면 p-value 계산 불가 → nan
    if all(np.all(g == g[0]) for g in groups):
        return np.nan
    return f_oneway(*groups).pvalue

def ch_index(vals, labels):
    """Calinski-Harabasz Index"""
    vals = np.asarray(vals).reshape(-1, 1)
    # 전부 같은 값이면 정의 무의미 → nan
    if np.allclose(vals, vals[0]):
        return np.nan
    return calinski_harabasz_score(vals, labels)

def cluster_robust_wald(vals, labels, clusters):
    """One-way cluster-robust Wald test for equality of class means."""
    from scipy.stats import chi2

    vals = np.asarray(vals, dtype=float)
    labels = np.asarray(labels)
    clusters = np.asarray(clusters)

    valid = np.isfinite(vals)
    vals = vals[valid]
    labels = labels[valid]
    clusters = clusters[valid]

    classes = np.unique(labels)
    unique_clusters = np.unique(clusters)
    if len(classes) < 2 or len(unique_clusters) < 3:
        return np.nan, np.nan
    if np.allclose(vals, vals[0]):
        return np.nan, np.nan

    n = len(vals)
    X = np.ones((n, len(classes)))
    for j, cls in enumerate(classes[1:], start=1):
        X[:, j] = (labels == cls).astype(float)

    p = X.shape[1]
    q = p - 1
    if n <= p or np.linalg.matrix_rank(X) < p or len(unique_clusters) <= q:
        return np.nan, np.nan

    xtx_inv = np.linalg.pinv(X.T @ X)
    beta_hat = xtx_inv @ X.T @ vals
    resid = vals - X @ beta_hat

    meat = np.zeros((p, p))
    for g in unique_clusters:
        msk = clusters == g
        score_g = X[msk].T @ resid[msk]
        meat += np.outer(score_g, score_g)

    cov = xtx_inv @ meat @ xtx_inv
    cov *= (len(unique_clusters) / (len(unique_clusters) - 1)) * ((n - 1) / (n - p))

    R = np.zeros((q, p))
    R[:, 1:] = np.eye(q)
    diff = R @ beta_hat
    cov_restricted = R @ cov @ R.T

    if not np.all(np.isfinite(cov_restricted)) or np.linalg.matrix_rank(cov_restricted) < q:
        return np.nan, np.nan

    wald_stat = float(diff.T @ np.linalg.pinv(cov_restricted) @ diff)
    if wald_stat < 0 and np.isclose(wald_stat, 0):
        wald_stat = 0.0
    p_value = float(chi2.sf(wald_stat, q))
    return wald_stat, p_value

metrics = {
    "FDR"      : fisher_discriminant_ratio,
    "ANOVA-F"  : anova_F,
    "ANOVA-p"  : anova_p_value,
    "CH"       : ch_index,
}

def fmt(x):
    # nan → 0.000으로 출력
    return f"{0.0:8.3f}" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{x:8.3f}"

def fmt_p(name, x):
    if name == "ANOVA-p":
        return f"{0.0:10.3e}" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{x:10.3e}"
    return fmt(x)

def fmt_wald(x):
    return f"{'nan':>8}" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{x:8.3f}"

def print_block(title, init_vals, train_vals, labels, file_ids=None):
    """공통 블록 출력: Global + Per-file"""
    init_vals  = np.asarray(init_vals)
    train_vals = np.asarray(train_vals)

    print(f"\n\n==========  {title} 분리도 지표  ==========")

    # ── Global ──
    print("── Global ──")
    print(f"{'Metric':<10} | {'Init':>8} | {'Trained':>8}")
    print("-" * 34)
    for name, fn in metrics.items():
        g0 = fn(init_vals,  labels)
        g1 = fn(train_vals, labels)
        print(f"{name:<10} | {fmt_p(name, g0)} | {fmt_p(name, g1)}")

def print_block(title, init_vals, train_vals, labels, file_ids=None, cluster_ids=None):
    init_vals  = np.asarray(init_vals)
    train_vals = np.asarray(train_vals)
    labels     = np.asarray(labels)

    print(f"\n\n==========  {title} 분리도 지표  ==========")

    # ── Global ──
    print("── Global ──")
    print(f"{'Metric':<8} | {'Init':>8} | {'Trained':>8}")
    print("-" * 32)
    for name, fn in metrics.items():
        g0 = fn(init_vals,  labels)
        g1 = fn(train_vals, labels)
        print(f"{name:<8} | {fmt_p(name, g0)} | {fmt_p(name, g1)}")

    if cluster_ids is not None and len(cluster_ids) == len(train_vals):
        cluster_ids = np.asarray(cluster_ids)
        w0, p0 = cluster_robust_wald(init_vals, labels, cluster_ids)
        w1, p1 = cluster_robust_wald(train_vals, labels, cluster_ids)
        print(f"{'CR-Wald':<10} | {fmt_wald(w0)} | {fmt_wald(w1)}")
        print(f"{'CR-Wald-p':<10} | {fmt_wald(p0)} | {fmt_wald(p1)}")

    # ── Per-file ──
    if file_ids is not None and len(file_ids) == len(train_vals):

        unique_files = np.unique(file_ids)

        # 클래스 수가 파일 수와 같으면 (각 파일이 한 클래스)
        # 파일별 분리도 계산은 의미 없음 → 자동 스킵
        if len(unique_files) == len(np.unique(labels)):
            print("\n[Per-file evaluation skipped: each file corresponds to a single class]")
            return

        for f in unique_files:
            msk = (file_ids == f)

            # 파일 안에 클래스가 2개 이상일 때만 계산
            if len(np.unique(labels[msk])) < 2:
                continue

            print(f"\n── File {f} ──")
            print(f"{'Metric':<10} | {'Init':>8} | {'Trained':>8}")
            print("-" * 34)

            for name, fn in metrics.items():
                v0 = fn(init_vals[msk],  labels[msk])
                v1 = fn(train_vals[msk], labels[msk])
                print(f"{name:<10} | {fmt_p(name, v0)} | {fmt_p(name, v1)}")


labels_arr = np.array(cat_tags_all)
file_ids   = np.array(file_ids_all)
cluster_ids = np.array([f"{lab}_{fid}" for lab, fid in zip(labels_arr, file_ids)])

# Alpha 분리도 지표 출력 (Global + 파일별)
vals_alpha_init  = np.array(alpha_init_all)
vals_alpha_train = np.array(alpha_all)
print_block("Alpha", vals_alpha_init, vals_alpha_train, labels_arr, file_ids, cluster_ids)

# Beta 분리도 지표 출력 (Global + 파일별)
vals_beta_init  = np.array(beta_init_all)
vals_beta_train = np.array(beta_all)
print_block("Beta", vals_beta_init, vals_beta_train, labels_arr, file_ids, cluster_ids)

# Gamma 분리도 지표 출력 (Global + 파일별)
vals_r_init  = np.array(r_init_all)
vals_r_train = np.array(r_all)
print_block("Gamma", vals_r_init, vals_r_train, labels_arr, file_ids, cluster_ids)

# B 분리도 지표 출력 (Global + 파일별)
vals_B_init      = np.array(B_init_all)
vals_B_train    = np.array(B_all)
print_block("B", vals_B_init, vals_B_train, labels_arr, file_ids, cluster_ids)

# Omega 분리도 지표 출력 (Global + 파일별)
vals_omega_init      = np.array(omega_init_all)
vals_omega_train    = np.array(omega_all)
print_block("Omega", vals_omega_init, vals_omega_train, labels_arr, file_ids, cluster_ids)

**Evaluation Metric without RUN**

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import f_oneway, chi2
from sklearn.metrics import calinski_harabasz_score

# 저장된 모델 파라미터 CSV만 사용해서 Evaluation Metric 재계산
CSV_PATH = PARAM_CSV_PATH
params_df = pd.read_csv(CSV_PATH)

required_cols = {"category", "file_idx", "alpha", "beta", "omega", "r", "B"}
missing_cols = required_cols - set(params_df.columns)
if missing_cols:
    raise ValueError(f"CSV에 필요한 컬럼이 없습니다: {sorted(missing_cols)}")

labels_arr = params_df["category"].to_numpy()
file_ids = params_df["file_idx"].to_numpy()
cluster_ids = np.array([f"{lab}_{fid}" for lab, fid in zip(labels_arr, file_ids)])
n = len(params_df)

# train_one_window / Evaluation Metric 셀에서 쓰던 초기값과 동일하게 설정
alpha0 = 0.5
r0 = 1.0
omega0 = 2.0 * np.pi
beta0 = omega0**2
B0 = 0.5


def fisher_discriminant_ratio(vals, labels):
    """FDR: (클래스 평균 분산) / (클래스 내부 분산)"""
    vals = np.asarray(vals, dtype=float)
    labels = np.asarray(labels)
    classes = np.unique(labels)
    means = [vals[labels == c].mean() for c in classes]
    variances = [vals[labels == c].var(ddof=1) for c in classes]
    between = np.var(means, ddof=1)
    within = np.mean(variances)
    return between / within if within > 0 else np.nan


def anova_F(vals, labels):
    """ANOVA F-statistic"""
    vals = np.asarray(vals, dtype=float)
    labels = np.asarray(labels)
    groups = [vals[labels == c] for c in np.unique(labels)]
    if any(len(g) < 2 for g in groups):
        return np.nan
    if all(np.all(g == g[0]) for g in groups):
        return np.nan
    return f_oneway(*groups).statistic


def anova_p_value(vals, labels):
    """ANOVA p-value"""
    vals = np.asarray(vals, dtype=float)
    labels = np.asarray(labels)
    groups = [vals[labels == c] for c in np.unique(labels)]
    if any(len(g) < 2 for g in groups):
        return np.nan
    if all(np.all(g == g[0]) for g in groups):
        return np.nan
    return f_oneway(*groups).pvalue


def ch_index(vals, labels):
    """Calinski-Harabasz Index"""
    vals = np.asarray(vals, dtype=float).reshape(-1, 1)
    if np.allclose(vals, vals[0]):
        return np.nan
    return calinski_harabasz_score(vals, labels)


def cluster_robust_wald(vals, labels, clusters):
    """One-way cluster-robust Wald test for equality of class means."""
    vals = np.asarray(vals, dtype=float)
    labels = np.asarray(labels)
    clusters = np.asarray(clusters)

    valid = np.isfinite(vals)
    vals = vals[valid]
    labels = labels[valid]
    clusters = clusters[valid]

    classes = np.unique(labels)
    unique_clusters = np.unique(clusters)
    if len(classes) < 2 or len(unique_clusters) < 3:
        return np.nan, np.nan
    if np.allclose(vals, vals[0]):
        return np.nan, np.nan

    n_obs = len(vals)
    X = np.ones((n_obs, len(classes)))
    for j, cls in enumerate(classes[1:], start=1):
        X[:, j] = (labels == cls).astype(float)

    p = X.shape[1]
    q = p - 1
    if n_obs <= p or np.linalg.matrix_rank(X) < p or len(unique_clusters) <= q:
        return np.nan, np.nan

    xtx_inv = np.linalg.pinv(X.T @ X)
    beta_hat = xtx_inv @ X.T @ vals
    resid = vals - X @ beta_hat

    meat = np.zeros((p, p))
    for g in unique_clusters:
        msk = clusters == g
        score_g = X[msk].T @ resid[msk]
        meat += np.outer(score_g, score_g)

    cov = xtx_inv @ meat @ xtx_inv
    cov *= (len(unique_clusters) / (len(unique_clusters) - 1)) * ((n_obs - 1) / (n_obs - p))

    R = np.zeros((q, p))
    R[:, 1:] = np.eye(q)
    diff = R @ beta_hat
    cov_restricted = R @ cov @ R.T

    if not np.all(np.isfinite(cov_restricted)) or np.linalg.matrix_rank(cov_restricted) < q:
        return np.nan, np.nan

    wald_stat = float(diff.T @ np.linalg.pinv(cov_restricted) @ diff)
    if wald_stat < 0 and np.isclose(wald_stat, 0):
        wald_stat = 0.0
    p_value = float(chi2.sf(wald_stat, q))
    return wald_stat, p_value


metrics = {
    "FDR": fisher_discriminant_ratio,
    "ANOVA-F": anova_F,
    "ANOVA-p": anova_p_value,
    "CH": ch_index,
}


def fmt(x):
    return f"{0.0:8.3f}" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{x:8.3f}"


def fmt_p(name, x):
    if name == "ANOVA-p":
        return f"{0.0:10.3e}" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{x:10.3e}"
    return fmt(x)


def fmt_wald(x):
    return f"{'nan':>8}" if (x is None or (isinstance(x, float) and np.isnan(x))) else f"{x:8.3f}"


def print_block(title, init_vals, train_vals, labels, file_ids=None, cluster_ids=None):
    init_vals = np.asarray(init_vals)
    train_vals = np.asarray(train_vals)
    labels = np.asarray(labels)

    print(f"\n\n==========  {title} 분리도 지표  ==========")

    print("── Global ──")
    print(f"{'Metric':<8} | {'Init':>10} | {'Trained':>10}")
    print("-" * 38)
    for name, fn in metrics.items():
        g0 = fn(init_vals, labels)
        g1 = fn(train_vals, labels)
        print(f"{name:<8} | {fmt_p(name, g0)} | {fmt_p(name, g1)}")

    if cluster_ids is not None and len(cluster_ids) == len(train_vals):
        w0, p0 = cluster_robust_wald(init_vals, labels, cluster_ids)
        w1, p1 = cluster_robust_wald(train_vals, labels, cluster_ids)
        print(f"{'CR-Wald':<10} | {fmt_wald(w0)} | {fmt_wald(w1)}")
        print(f"{'CR-Wald-p':<10} | {fmt_wald(p0)} | {fmt_wald(p1)}")

    if file_ids is not None and len(file_ids) == len(train_vals):
        unique_files = np.unique(file_ids)
        if len(unique_files) == len(np.unique(labels)):
            print("\n[Per-file evaluation skipped: each file corresponds to a single class]")
            return

        for f in unique_files:
            msk = file_ids == f
            if len(np.unique(labels[msk])) < 2:
                continue

            print(f"\n── File {f} ──")
            print(f"{'Metric':<10} | {'Init':>10} | {'Trained':>10}")
            print("-" * 40)
            for name, fn in metrics.items():
                v0 = fn(init_vals[msk], labels[msk])
                v1 = fn(train_vals[msk], labels[msk])
                print(f"{name:<10} | {fmt_p(name, v0)} | {fmt_p(name, v1)}")


print(f"[LOAD] {CSV_PATH}")
print(f"[INFO] n={n}, categories={params_df['category'].value_counts().to_dict()}, files={sorted(params_df['file_idx'].unique().tolist())}")

print_block("Alpha", np.full(n, alpha0), params_df["alpha"].to_numpy(), labels_arr, file_ids, cluster_ids)
print_block("Beta", np.full(n, beta0), params_df["beta"].to_numpy(), labels_arr, file_ids, cluster_ids)
print_block("Gamma", np.full(n, r0), params_df["r"].to_numpy(), labels_arr, file_ids, cluster_ids)
print_block("B", np.full(n, B0), params_df["B"].to_numpy(), labels_arr, file_ids, cluster_ids)
print_block("Omega", np.full(n, omega0), params_df["omega"].to_numpy(), labels_arr, file_ids, cluster_ids)


**Saved Model Load**

In [ ]:
def _parse_model_filename(fn: str):
    """
    지원 파일명 예:
      model_3_140_2b2_Tc_mc_LittC2SE_Crackle_window_1.pt

    규칙:
      model_{file_idx}_{category}_window_{win}.pt
      - file_idx에는 '_'가 얼마든지 포함될 수 있음
      - category는 window 바로 앞의 단일 토큰
    """
    if not (fn.startswith("model_") and fn.endswith(".pt")):
        return None, None, None

    name = fn[len("model_"):-len(".pt")]  # model_, .pt 제거

    if "_window_" not in name:
        return None, None, None

    left, win_str = name.rsplit("_window_", 1)

    try:
        win = int(win_str)
    except ValueError:
        return None, None, None

    # left = "{file_idx}_{category}"
    if "_" not in left:
        return None, None, None

    file_idx, cat = left.rsplit("_", 1)

    return file_idx, cat, win


def export_pinn_params_to_csv(
    models_dir=SAVED_MODEL_DIR,
    csv_path=PARAM_CSV_PATH,
    ModelClass=MlpModel,   
    cfg=None,
    device="cpu",
    win_sec=None,       # ✅ 학습 때 WIN_SEC 값을 여기로 전달 (전역 WIN_SEC를 import해서 넣어도 됨)
):
    """
    models_dir: 저장된 .pt들이 있는 폴더
    csv_path: 출력 csv 경로
    ModelClass, cfg: 모델을 '동일 구조'로 재생성하기 위한 정보 (필수)
      - 예: model = ModelClass(cfg).to(device)

    반환:
      alpha_list, beta_list, omega_list, r_list, B_list, cat_tags, win_ids, rows
    """
    if ModelClass is None or cfg is None:
        raise ValueError("ModelClass와 cfg를 넘겨주셔야 합니다.")
    if win_sec is None:
        raise ValueError("win_sec를 반드시 넘겨주셔야 합니다. (예: win_sec=WIN_SEC)")


    omega_init = 2.0 * np.pi
    beta_init  = omega_init**2
    B_init     = 0.5
    scaling_factor = 1.0 / win_sec

    # ---- 기존 변수명 유지 ----
    alpha_list = []
    beta_list  = []
    omega_list = []
    r_list     = []
    B_list     = []
    phi_list   = [] 
    cat_tags   = []
    win_ids    = []  # 카테고리 내부 window 번호(파일명에서 읽은 win)

    alpha_hat_list = []
    beta_hat_list  = []
    omega_hat_list = []
    r_hat_list     = []
    B_hat_list     = []

    rows = []

    fns = sorted(
    [fn for fn in os.listdir(models_dir) if fn.endswith(".pt") and fn.startswith("model_")],
    key=lambda fn: (
        _parse_model_filename(fn)[0],  # file_idx
        _parse_model_filename(fn)[1],  # category
        _parse_model_filename(fn)[2],  # window
        )
    )   


    for fn in fns:
        file_idx, cat, win = _parse_model_filename(fn)
        if file_idx is None:
            # 규칙 안 맞는 파일은 스킵
            continue

        model_path = os.path.join(models_dir, fn)

        # 모델 재생성 + state_dict 로드
        model = ModelClass(
                    in_dim=1,
                    n_hidden=cfg.n_hidden,
                    n_neurons=cfg.n_neurons,
                    beta_init=beta_init, # dummy
                    omega_init=omega_init, # dummy
                    B_init=B_init, # dummy
                    fourier_m=8,
                    fourier_sigma=1.0,
                    use_fourier=False,
                    scaling_factor=scaling_factor # 반드시 학습 때와 동일
                ).to(device)

        sd = torch.load(model_path, map_location=device)
        model.load_state_dict(sd)
        model.eval()

        # 파라미터 추출
        # 물리 복원 파라미터
        alpha = float(model.alpha.item())
        beta  = float(model.beta.item())
        omega = float(model.omega.item())
        r     = float(model.r.item())
        B     = float(model.B.item())
        phi   = float(model.phi.item())

        # hat 파라미터
        alpha_hat = float(model.alpha_hat.item())
        beta_hat  = float(model.beta_hat.item())
        omega_hat = float(model.omega_hat.item())
        r_hat     = float(model.r_hat.item())
        B_hat     = float(model.B_hat.item())


        # 기존 리스트들에 append
        alpha_list.append(alpha)
        beta_list.append(beta)
        omega_list.append(omega)
        r_list.append(r)
        B_list.append(B)
        phi_list.append(phi)
        cat_tags.append(cat)
        win_ids.append(win)

        alpha_hat_list.append(alpha_hat)
        beta_hat_list.append(beta_hat)
        omega_hat_list.append(omega_hat)
        r_hat_list.append(r_hat)
        B_hat_list.append(B_hat)

        rows.append({
            "model_file": fn,
            "model_path": model_path,
            "file_idx": file_idx,
            "category": cat,
            "window": win,
            "win_sec": float(win_sec),
            "scaling_factor": float(scaling_factor),

            # 물리 복원 파라미터
            "alpha": alpha,
            "beta": beta,
            "omega": omega,
            "r": r,
            "B": B,
            "phi": phi,
            
            # hat 파라미터
            "alpha_hat": alpha_hat,
            "beta_hat": beta_hat,
            "omega_hat": omega_hat,
            "r_hat": r_hat,
            "B_hat": B_hat,
        })

        # 메모리 정리
        del model
        torch.cuda.empty_cache()


    # CSV 저장 (pandas 없이도 되게 csv 모듈 사용)
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "model_file","model_path","file_idx","category","window",
                "win_sec","scaling_factor","alpha","beta","omega","r","B","phi",
                "alpha_hat","beta_hat","omega_hat","r_hat","B_hat",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"[DONE] Saved CSV -> {csv_path} (n_models={len(rows)})")

    return alpha_list, beta_list, omega_list, r_list, B_list, phi_list, alpha_hat_list, beta_hat_list, omega_hat_list, r_hat_list, B_hat_list, cat_tags, win_ids, rows


# call
alpha_list, beta_list, omega_list, r_list, B_list, phi_list, \
alpha_hat_list, beta_hat_list, omega_hat_list, r_hat_list, B_hat_list, \
cat_tags, win_ids, rows = export_pinn_params_to_csv(
    models_dir=SAVED_MODEL_DIR,
    csv_path=PARAM_CSV_PATH,
    ModelClass=MlpModel,
    cfg=cfg,
    device="cpu",
    win_sec=WIN_SEC
)

**ReLU: GT vs PINN vs ODE (w/o u'')**

In [ ]:
models_dir = Path(SAVED_MODEL_DIR)
params_csv_path = Path(PARAM_CSV_PATH)
pdf_dir = RESULTS_DIR / 'ode_forward_results'
pdf_dir.mkdir(parents=True, exist_ok=True)

WIN_SEC_LOCAL = globals().get('WIN_SEC', 0.1)
scaling_factor = 1.0 / WIN_SEC_LOCAL
n_plot = 10
plot_each_window = True
plot_window_chunks = True
plot_full_signal = True

device = cfg.device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')

required_window_vars = ['window_H', 'window_W', 'window_C']
missing_window_vars = [name for name in required_window_vars if name not in globals()]
if missing_window_vars:
    raise NameError(
        'Run the data/window preprocessing cells first. Missing: '
        + ', '.join(missing_window_vars)
    )

required_cols = {
    'file_idx', 'category', 'window', 'model_path',
    'alpha_hat', 'beta_hat', 'omega_hat', 'r_hat', 'B_hat', 'phi'
}


def rebuild_param_csv_from_saved_models():
    if 'export_pinn_params_to_csv' not in globals():
        raise RuntimeError(
            'Parameter CSV is missing/incomplete and export_pinn_params_to_csv() is not defined. '
            'Run the Saved Model Load/export cell first, then rerun this plot cell.'
        )

    return export_pinn_params_to_csv(
        models_dir=str(models_dir),
        csv_path=str(params_csv_path),
        ModelClass=MlpModel,
        cfg=cfg,
        device='cpu',
        win_sec=WIN_SEC_LOCAL,
    )


if not params_csv_path.exists():
    print(f'[INFO] CSV not found. Rebuilding from saved models: {params_csv_path}')
    rebuild_param_csv_from_saved_models()

df_params = pd.read_csv(params_csv_path)
missing_cols = required_cols - set(df_params.columns)
if missing_cols:
    print(f'[INFO] CSV is missing {sorted(missing_cols)}. Rebuilding from saved models.')
    rebuild_param_csv_from_saved_models()
    df_params = pd.read_csv(params_csv_path)

df_params['file_idx'] = df_params['file_idx'].astype(int)
df_params['window'] = df_params['window'].astype(int)


def relu_no_utt_forward_saved(t, u0, alpha, beta, r, B, omega, phi=0.0,
                              method='RK45', rtol=1e-7, atol=1e-9):
    """
    ReLU-degenerate forward model without u'':
        alpha*u_tau + beta*u + r*u^3 = B*cos(omega*tau + phi)

    This matches the ODE forward method used in the markdown-adjacent
    GT vs ODE (w/o u'') cell. Only u0 is used as the initial condition.
    """
    t = np.asarray(t, dtype=np.float64).ravel()
    alpha = float(alpha)
    beta = float(beta)
    r = float(r)
    B = float(B)
    omega = float(omega)
    phi = float(phi)

    if np.isclose(alpha, 0.0):
        raise RuntimeError("alpha is too close to zero for ReLU no-u'' first-order forward")

    def rhs(ti, y):
        u = y[0]
        du = (B * np.cos(omega * ti + phi) - beta * u - r * u**3) / alpha
        return [du]

    sol = solve_ivp(
        rhs,
        t_span=(float(t[0]), float(t[-1])),
        y0=[float(u0)],
        t_eval=t,
        method=method,
        rtol=rtol,
        atol=atol,
    )

    if not sol.success:
        raise RuntimeError(f"ReLU no-u'' ODE solve failed: {sol.message}")

    return sol.y[0]

def fd_initial_conditions_saved(t_raw, x_raw):
    t_raw = np.asarray(t_raw, dtype=np.float64).ravel()
    x_raw = np.asarray(x_raw, dtype=np.float64).ravel()
    dt = float(np.diff(t_raw).mean())
    u0 = float(x_raw[0])

    if len(x_raw) >= 3:
        v0_dt = (-3 * x_raw[0] + 4 * x_raw[1] - x_raw[2]) / (2 * dt)
    else:
        v0_dt = (x_raw[1] - x_raw[0]) / dt

    return u0, float(v0_dt / scaling_factor)  # du/dtau


def make_model_for_saved_state():
    omega_init = 2.0 * np.pi
    beta_init = omega_init**2
    B_init = 0.5

    return MlpModel(
        in_dim=1,
        n_hidden=cfg.n_hidden,
        n_neurons=cfg.n_neurons,
        beta_init=beta_init,
        omega_init=omega_init,
        B_init=B_init,
        fourier_m=3,
        fourier_sigma=1.0,
        use_fourier=False,
        scaling_factor=scaling_factor,
    )


def load_saved_model(model_path):
    model = make_model_for_saved_state().to(device)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def get_param_row(file_idx, category_name, window):
    row = df_params[
        (df_params['file_idx'] == int(file_idx)) &
        (df_params['category'] == category_name) &
        (df_params['window'] == int(window))
    ]
    return None if row.empty else row.iloc[0]


def model_path_from_row(row, file_idx, category_name, window):
    if row is not None and 'model_file' in row.index and pd.notna(row['model_file']):
        return models_dir / str(row['model_file'])
    if row is not None and 'model_path' in row.index and pd.notna(row['model_path']):
        model_path = Path(row['model_path'])
        return model_path if not model_path.is_absolute() else models_dir / model_path.name
    return models_dir / f'model_{file_idx}_{category_name}_window_{window}.pt'


def save_current_fig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if 'save_plot_as_pdf' in globals():
        save_plot_as_pdf(str(path))
    else:
        plt.savefig(path, format='pdf', bbox_inches='tight')


def apply_train_prediction_plot_format(title_msg):
    """Match the prediction plot format used inside train_all_windows()."""
    plt.xlabel('Time', fontsize=14)
    plt.ylabel('Amplitude', fontsize=14)
    plt.title(title_msg, fontsize=14)
    plt.legend(fontsize=12)
    plt.tick_params(axis='both', which='major', labelsize=14)
    ax = plt.gca()
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    plt.tight_layout()


file_windows_by_category = {
    'Healthy': window_H,
    'Wheeze': window_W,
    'Crackle': window_C,
}

ic_rows = []
missing_param_rows = []
missing_model_paths = []

print(f'[CSV LOAD] {params_csv_path}')
print(f'[CSV ROWS] {len(df_params)} parameter rows')
print(f'[MODEL DIR] {models_dir}')
print(f'[SAVE DIR] {pdf_dir}')
print("[ODE MODEL] ReLU no-u'': alpha*u_tau + beta*u + r*u^3 = B*cos(omega*tau + phi)")

for category_name, file_windows in file_windows_by_category.items():
    for file_idx, file_data in enumerate(file_windows, start=1):
        print(f'\n[FILE] {category_name} file={file_idx}')

        t_file_all, gt_file_all, pinn_file_all, ode_file_all = [], [], [], []

        t_windows, x_windows = file_data[:2]
        x_windows_raw = np.asarray(x_windows)  # raw x_windows: no x_maxabs/z-score normalization

        for win_zero_based in range(len(t_windows)):
            window = win_zero_based + 1
            row = get_param_row(file_idx, category_name, window)
            if row is None:
                missing_param_rows.append((category_name, file_idx, window))
                continue

            model_path = model_path_from_row(row, file_idx, category_name, window)
            if not model_path.exists():
                missing_model_paths.append(str(model_path))
                continue

            t_raw = np.asarray(t_windows[win_zero_based], dtype=np.float64).ravel()
            x_raw = np.asarray(x_windows_raw[win_zero_based], dtype=np.float64).ravel()
            tau_local = scaling_factor * t_raw
            t_plot = t_raw + win_zero_based * WIN_SEC_LOCAL

            model = load_saved_model(model_path)
            with torch.no_grad():
                tau_tensor = torch.as_tensor(
                    tau_local[:, None],
                    dtype=torch.float32,
                    device=device,
                )
                u_pinn = model(tau_tensor).detach().cpu().numpy().squeeze()

                # Use the parameters from the loaded model so PINN and ODE always match the same checkpoint.
                alpha_hat = float(model.alpha_hat.item())
                beta_hat = float(model.beta_hat.item())
                omega_hat = float(model.omega_hat.item())
                r_hat = float(model.r_hat.item())
                B_hat = float(model.B_hat.item())
                phi = float(model.phi.item())

            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()

            u0, v0_tau = fd_initial_conditions_saved(t_raw, x_raw)  # v0_tau is diagnostics only; no-u'' ODE uses u0.

            try:
                u_ode = relu_no_utt_forward_saved(
                    t=tau_local,
                    u0=u0,
                    alpha=alpha_hat,
                    beta=beta_hat,
                    r=r_hat,
                    B=B_hat,
                    omega=omega_hat,
                    phi=phi,
                )
                ode_success = True
                ode_fail_msg = ''
            except RuntimeError as e:
                u_ode = np.full_like(x_raw, np.nan)
                ode_success = False
                ode_fail_msg = str(e)
                print(f'[ODE FAIL] {category_name} file={file_idx} win={window} | {ode_fail_msg}')

            t_file_all.append(t_plot)
            gt_file_all.append(x_raw)
            pinn_file_all.append(u_pinn)
            ode_file_all.append(u_ode)

            if window <= n_plot:
                ic_rows.append({
                    'category': category_name,
                    'file_idx': file_idx,
                    'window': window,
                    'u0': u0,
                    'v0_tau': v0_tau,
                    'alpha_hat': alpha_hat,
                    'beta_hat': beta_hat,
                    'omega_hat': omega_hat,
                    'r_hat': r_hat,
                    'B_hat': B_hat,
                    'phi': phi,
                    'model_path': str(model_path),
                    'ode_success': ode_success,
                    'ode_fail_msg': ode_fail_msg,
                })

                print(
                    f'[IC] {category_name} | file={file_idx} | win={window} | '
                    f'u0={u0:.6e}, v0_tau={v0_tau:.6e}'
                )

                if plot_each_window:
                    plt.figure(figsize=(8, 3))
                    plt.plot(t_plot, u_pinn, label='PINN pred')
                    plt.plot(t_plot, x_raw, '--', label='Ground Truth')
                    plt.plot(t_plot, u_ode, color='green', label="ODE forward (ReLU w/o u'')")
                    title_msg = f"{category_name} window-{window} prediction | ODE (w/o u'')"
                    if not ode_success:
                        title_msg += ' | ODE FAIL'
                    apply_train_prediction_plot_format(title_msg)

                    pdf_filename = pdf_dir / f'{category_name}_GT_PINN_ODE_file{file_idx}_win{window:03d}.pdf'
                    save_current_fig(pdf_filename)
                    print(f'[SAVE] {pdf_filename}')
                    plt.show()
                    plt.close()

        if plot_window_chunks and t_file_all:
            n_windows_file = len(t_file_all)
            for chunk_start in range(0, n_windows_file, n_plot):
                chunk_end = min(chunk_start + n_plot, n_windows_file)
                first_window = chunk_start + 1
                last_window = chunk_end

                t_chunk = np.concatenate(t_file_all[chunk_start:chunk_end])
                gt_chunk = np.concatenate(gt_file_all[chunk_start:chunk_end])
                pinn_chunk = np.concatenate(pinn_file_all[chunk_start:chunk_end])
                ode_chunk = np.concatenate(ode_file_all[chunk_start:chunk_end])

                plt.figure(figsize=(8, 3))
                plt.plot(t_chunk, pinn_chunk, label='PINN pred')
                plt.plot(t_chunk, gt_chunk, '--', label='Ground Truth')
                plt.plot(t_chunk, ode_chunk, color='green', label="ODE forward (ReLU w/o u'')")
                apply_train_prediction_plot_format(
                    f"{category_name} windows {first_window}-{last_window} prediction | ODE (w/o u'')"
                )

                pdf_filename = pdf_dir / f'{category_name}_GT_PINN_ODE_file{file_idx}_win{first_window:03d}-{last_window:03d}.pdf'
                save_current_fig(pdf_filename)
                print(f'[SAVE FILE PLOT] {pdf_filename}')
                plt.show()
                plt.close()

        if plot_full_signal and t_file_all:
            t_all = np.concatenate(t_file_all)
            gt_all = np.concatenate(gt_file_all)
            pinn_all = np.concatenate(pinn_file_all)
            ode_all = np.concatenate(ode_file_all)

            plt.figure(figsize=(8, 3))
            plt.plot(t_all, pinn_all, label='PINN pred')
            plt.plot(t_all, gt_all, '--', label='Ground Truth')
            plt.plot(t_all, ode_all, color='green', label="ODE forward (ReLU w/o u'')")
            apply_train_prediction_plot_format(
                f"{category_name} full signal prediction | ODE (w/o u'')"
            )

            pdf_filename = pdf_dir / f'{category_name}_GT_PINN_ODE_file{file_idx}_FULL.pdf'
            save_current_fig(pdf_filename)
            print(f'[SAVE FILE PLOT] {pdf_filename}')
            plt.show()
            plt.close()

ic_csv_path = pdf_dir / 'GT_PINN_ODE_initial_conditions.csv'
if ic_rows:
    pd.DataFrame(ic_rows).to_csv(ic_csv_path, index=False)
    print(f'[DONE] Saved initial conditions CSV -> {ic_csv_path}')

if missing_param_rows:
    print(f'[WARN] Missing parameter rows: {len(missing_param_rows)}')
    print(missing_param_rows[:10])

if missing_model_paths:
    print(f'[WARN] Missing model files: {len(missing_model_paths)}')
    print(missing_model_paths[:10])

print("[DONE] ReLU GT vs PINN vs ODE (w/o u'') plots generated from 0514 saved results")